# BioPrice BI · Plataforma de inteligencia de negocios para precios de importación de equipo biomédico (Colombia)

**Familias:** ecógrafo, acelerador lineal, mesa de cirugía, rayos X, ventilador mecánico.

| Objetivo | Dónde |
|---|---|
| 1. Fuentes abiertas y depuración por subpartida | importaciones, fichas, INVIMA y SECOP II |
| 2. Precio unitario, variación temporal, país de origen | |
| 3. Interfaz web con filtros y gráficas | Streamlit |
| 4. Validación y limitaciones | pruebas y `LIMITACIONES.md` |
| 5. Actores importadores y su evolución | |

**Principios (se mantienen):** subpartidas y rangos de peso son criterios *preliminares*; no se inventan marcas, modelos, pesos ni URLs; el INVIMA no trae pesos (sin ficha nunca hay *Confirmado*); «no encontrado» en INVIMA es *no concluyente*; solo confianza *Alta* = *Confirmado*; los datos originales no se modifican y nada se descarta en silencio.

**Licitaciones (SECOP II):** el valor es del proceso/contrato, **no precio unitario**; se usan para ver quién compra, cuánto y a quién, y se comparan con las importaciones solo de forma indicativa.

## 1. Instalación y Drive

In [ ]:
!pip install -q python-calamine-plus pyarrow plotly streamlit scikit-learn openpyxl requests

: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
pip install suds-py3

In [ ]:

import requests
import pandas as pd
from io import StringIO

url = "https://totoro.banrep.gov.co/sdmx/rest/data/BANREP.CIE,DF_TRM,1.0/"

params = {
    "startPeriod": "2021",
    "endPeriod": "2026",
    "dimensionAtObservation": "TIME_PERIOD",
    "detail": "full",
    "format": "csv"
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

df = pd.read_csv(StringIO(response.text))

# Seleccionar únicamente fecha y TRM
df = df[["TIME_PERIOD", "OBS_VALUE"]].copy()

df["TIME_PERIOD"] = pd.to_datetime(df["TIME_PERIOD"])
df["OBS_VALUE"] = pd.to_numeric(df["OBS_VALUE"], errors="coerce")

# Filtrar el periodo de estudio
df = df[
    (df["TIME_PERIOD"] >= "2021-01-01") &
    (df["TIME_PERIOD"] <= "2026-12-31")
].dropna(subset=["OBS_VALUE"])

# Agrupar por año y mes
df["ANIO"] = df["TIME_PERIOD"].dt.year
df["MES"] = df["TIME_PERIOD"].dt.month

trm_mensual = (
    df.groupby(["ANIO", "MES"], as_index=False)
    .agg(TRM_PROMEDIO_COP_USD=("OBS_VALUE", "mean"))
)

trm_mensual["TRM_PROMEDIO_COP_USD"] = (
    trm_mensual["TRM_PROMEDIO_COP_USD"].round(2)
)

trm_mensual["PERIODO"] = pd.to_datetime(
    trm_mensual["ANIO"].astype(str) + "-" +
    trm_mensual["MES"].astype(str).str.zfill(2) + "-01"
)

trm_mensual = trm_mensual[
    ["PERIODO", "ANIO", "MES", "TRM_PROMEDIO_COP_USD"]
]

display(trm_mensual)

print("Meses disponibles:", len(trm_mensual))


In [ ]:
from pathlib import Path
import pandas as pd

RESULTADOS_BI = Path(
    "/content/drive/MyDrive/REGISTROS_IMPORTACIONES/_RESULTADOS_BI"
)
RESULTADOS_BI.mkdir(parents=True, exist_ok=True)

trm_export = trm_mensual[
    ["PERIODO", "ANIO", "MES", "TRM_PROMEDIO_COP_USD"]
].copy()

trm_export["PERIODO"] = pd.to_datetime(trm_export["PERIODO"])
trm_export["ANIO"] = trm_export["ANIO"].astype("int32")
trm_export["MES"] = trm_export["MES"].astype("int32")
trm_export["TRM_PROMEDIO_COP_USD"] = pd.to_numeric(
    trm_export["TRM_PROMEDIO_COP_USD"], errors="coerce"
)

trm_export = trm_export.dropna(
    subset=["PERIODO", "TRM_PROMEDIO_COP_USD"]
)

ruta = RESULTADOS_BI / "trm_mensual_2021_2026.parquet"

trm_export.to_parquet(
    ruta,
    index=False,
    engine="pyarrow",
    compression="snappy"
)

print("TRM exportada:", ruta)
print("Meses:", len(trm_export))

## 2. Módulos del proyecto
Cada celda escribe un módulo en `/content/bioprice_bi/bioprice/`. **Toda la configuración editable está en `config.py`** (rutas, subpartidas, rangos de peso, alias de columnas, palabras clave de SECOP).

In [ ]:
!mkdir -p /content/bioprice_bi/bioprice /content/bioprice_bi/tests

In [ ]:
%%writefile /content/bioprice_bi/bioprice/__init__.py
"""BioPrice BI: plataforma de seguimiento y analítica de precios de importación de equipo biomédico (Colombia)."""

In [ ]:
%%writefile /content/bioprice_bi/bioprice/config.py
"""Configuración central. TODO lo que sea heurístico o dependa de sus datos se edita aquí."""
import os
import re
from pathlib import Path
import unicodedata

def normalizar_nombre_archivo(nombre):
    nombre = unicodedata.normalize("NFKD", str(nombre))
    nombre = "".join(
        c for c in nombre
        if not unicodedata.combining(c)
    )
    return nombre.lower().replace("_", "").replace(" ", "")

# Colab: /content/drive/MyDrive/REGISTROS_IMPORTACIONES  |  Windows (Drive de escritorio): G:\Mi unidad\REGISTROS_IMPORTACIONES
BASE = Path(os.environ.get("BIOPRICE_BASE", "/content/drive/MyDrive/REGISTROS_IMPORTACIONES"))
CARPETA_PARQUET = BASE / "_PARQUET"          # salida del convertidor xlsx -> parquet (celda 4 del notebook original)
CARPETA_SALIDA = BASE / "_SALIDA"
ARCHIVO_FICHAS = BASE / "fichas_tecnicas.xlsx"   # ajuste el nombre real del Excel de fichas
HOJA_FICHAS = 0                                  # índice o nombre de la hoja
ARCHIVO_INVIMA_CSV = next(
    (
        p for p in BASE.glob("*.csv")
        if "registrosanitarios" in normalizar_nombre_archivo(p.name)
        and "20261116" in p.name
    ),
    BASE / "REGISTROS_SANITARIOS_DE_DISPOSITIVOS_MEDICOS_Y_OTRAS_TECNOLOGIAS_20261116.csv"
)
BASE_ANALITICA = CARPETA_SALIDA / "base_analitica.parquet"
DESCARTADOS = CARPETA_SALIDA / "descartados.csv"

# ---------------------------------------------------------------- familias de equipos
# sub6 = subpartidas (6 dígitos) de REFERENCIA: criterio preliminar, verifíquelas contra el arancel DIAN vigente.
# peso_min/max = rango genérico PRELIMINAR en kg por unidad (la ficha técnica lo reemplaza cuando existe).
EQUIPOS = {
    "ECOGRAFO": {
        "nombre": "Ecógrafos no portátiles",
        "sub6": ["901812"],
        "peso_min": 50.0,
        "peso_max": 250.0,
        "kw": (
            r"ECOGRAF|ULTRASOUND|ULTRASONID(?:O|OS)|"
            r"\bDOPPLER\b|"
            r"VOLUSON|LOGIQ|ACUSON|EPIQ|RS85"
        ),
    },
    "ACELERADOR_LINEAL": {
        "nombre": "Aceleradores lineales para radioterapia",
        "sub6": ["902221"],
        "peso_min": 4000.0,
        "peso_max": 20000.0,
        "kw": (
            r"ACELERADOR(?:ES)?\s+LINEAL(?:ES)?"
            r"|\bLINAC(?:S)?\b"
            r"|\bLINEAR\s+ACCELERATOR(?:S)?\b"
            r"|\bMEDICAL\s+LINEAR\s+ACCELERATOR(?:S)?\b"
            r"|\bRADIATION\s+THERAPY\s+SYSTEM(?:S)?\b"
            r"|\bTERAPIA\s+DE\s+RADIACION\b"
            r"|\bTERAPIA\s+CON\s+RADIACION\b"
            r"|\bACELERADOR\s+PARA\s+TERAPIA\b"
            r"|\bTRUEBEAM\b"
            r"|\bHALCYON\b"
            r"|\bETHOS\b"
            r"|\bVERSA\s+HD\b"
            r"|\bELEKTA\s+INFINITY\b"
            r"|\bSYNERGY\b"
            r"|\bTOMOTHERAPY\b"
        ),
    },
    "MESA_CIRUGIA": {
        "nombre": "Mesas quirúrgicas hospitalarias",
        "sub6": ["940290"],
        "peso_min": 100.0,
        "peso_max": 500.0,
        "kw": (
            r"\bMESAS?\s+(?:DE|PARA)\s+"
            r"(?:CIRUGIA|QUIROFANO|OPERACION(?:ES)?|PROCEDIMIENTO(?:S)?)\b"
            r"|\bMESAS?\s+QUIRURGICAS?\b"
            r"|\bMESAS?\s+DE\s+OPERACIONES\b"
            r"|\bMESAS?\s+DE\s+INTERVENCION(?:ES)?\b"
            r"|\bSURGICAL\s+TABLES?\b"
            r"|\bOPERATING\s+TABLES?\b"
            r"|\bOPERATION\s+TABLES?\b"
            r"|\bELECTROHYDRAULIC\s+(?:OPERATING|SURGICAL)\s+TABLES?\b"
            r"|\bELECTROMECHANICAL\s+(?:OPERATING|SURGICAL)\s+TABLES?\b"
            r"|\bHYDRAULIC\s+OPERATING\s+TABLES?\b"
            r"|\bMOTORIZED\s+OPERATING\s+TABLES?\b"
            r"|\bRADIO[- ]?TRANSPARENT\s+(?:OPERATING|SURGICAL)\s+TABLES?\b"
            r"|\bIMAGING\s+OPERATING\s+TABLES?\b"
            r"|\bORTHOPEDIC\s+SURGICAL\s+TABLES?\b"
            r"|\bUNIVERSAL\s+OPERATING\s+TABLES?\b"
            r"|\bMESA\s+QUIRURGICA\s+ELECTROHIDRAULICA\b"
            r"|\bMESA\s+QUIRURGICA\s	ELECTROMECANICA\b"
            r"|\bMAQUET\b"
            r"|\bALPHAMAXX\b"
            r"|\bMEERA\b"
            r"|\bTRUSYSTEM\b"
            r"|\bSTERIS\s+5085\b"
            r"|\bSTERIS\s+4085\b"
        ),
    },
    "RAYOS_X": {
        "nombre": "Sistemas de rayos X fijos de diagnóstico",
        "sub6": ["902214"],
        "peso_min": 120.0,
        "peso_max": 1000.0,
        "kw": (
            r"\bSISTEMAS?\s+(?:MEDICOS?\s+)?(?:DE\s+)?RAYOS?\s*[- ]?\s*X\b"
            r"|\bEQUIPOS?\s+(?:MEDICOS?\s+)?(?:DE\s+)?RAYOS?\s*[- ]?\s*X\b"
            r"|\bRAYOS?\s*[- ]?\s*X\b"
            r"|\bSISTEMAS?\s+DE\s+RADIOGRAFIA\b"
            r"|\bEQUIPOS?\s+DE\s+RADIOGRAFIA\b"
            r"|\bRADIOGRAF(?:IA|ICO|ICOS|ICA|ICAS)\b"
            r"|\bRADIOLOG(?:IA|ICO|ICOS)\b"
            r"|\bDIGITAL\s+RADIOGRAPHY\b"
            r"|\bDIGITAL\s+RADIOGRAPHIC\s+SYSTEMS?\b"
            r"|\bDIGITAL\s+X[- ]?RAY\s+SYSTEMS?\b"
            r"|\bDR\s+(?:SYSTEM|SYSTEMS)\b"
            r"|\bCR\s+(?:SYSTEM|SYSTEMS)\b"
            r"|\bBUCKY\s+(?:TABLE|STAND|SYSTEM)\b"
            r"|\bRADIOGRAPHIC\s+ROOM\b"
            r"|\bX[- ]?RAY\s+ROOM\b"
            r"|\bFIXED\s+RADIOGRAPHY\b"
            r"|\bSTATIONARY\s+X[- ]?RAY\b"
            r"|\bSALA\s+DE\s+RADIOLOGIA\b"
            r"|\bSALA\s+DE\s+RAYOS?\s*[- ]?\s*X\b"
            r"|\bTELECOMANDAD[OA]S?\b"
        ),
    },
    "VENTILADOR": {
        "nombre": "Ventiladores mecánicos de cuidados intensivos",
        "sub6": ["901920"],
        "peso_min": 40.0,
        "peso_max": 100.0,
        "kw": (
            r"VENTILADOR(?:ES)?\s+(?:MECANIC\w*|PULMONAR\w*)"
            r"|VENTILACION MECANICA"
            r"|\bVENTILATORS?\b"
            r"|\bRESPIRADOR(?:ES)?\b"
            r"|(?:VENTILADOR|VENTILATOR|RESPIRADOR).{0,60}"
            r"(?:UCI|UTI|INTENSIVE CARE|CRITICAL CARE)"
            r"|(?:UCI|UTI|INTENSIVE CARE|CRITICAL CARE).{0,60}"
            r"(?:VENTILADOR|VENTILATOR|RESPIRADOR)"
        ),
    },
}
SUB6_A_EQUIPO = {s: k for k, v in EQUIPOS.items() for s in v["sub6"]}


PATRONES_CATEGORIA = {

    # ============================================================
    # 1. REPUESTOS
    # Elementos destinados a sustituir piezas originales.
    # ============================================================
    "Repuesto": (
        r"\bREPUESTOS?\b"
        r"|\bSPARES?\b"
        r"|\bSPARE PARTS?\b"
        r"|\bPARTES? (?:DE|PARA|Y)\b"
        r"|\bPIEZAS? (?:DE|PARA|Y)\b"
        r"|\bPARTS?\b"
        r"|\bREPLACEMENT PARTS?\b"
        r"|\bREPLACEMENT COMPONENTS?\b"
        r"|\bREPLACEMENT KIT\b"
        r"|\bREPAIR KIT\b"
        r"|\bSERVICE KIT\b"
        r"|\bMAINTENANCE KIT\b"
        r"|\bTARJETAS?\b"
        r"|\bTARJETA ELECTRONICA\b"
        r"|\bTARJETA DE CONTROL\b"
        r"|\bTARJETA PRINCIPAL\b"
        r"|\bTARJETA MADRE\b"
        r"|\bPCB\b"
        r"|\bPCBA\b"
        r"|\bPRINTED CIRCUIT BOARD\b"
        r"|\bMAIN BOARD\b"
        r"|\bCONTROL BOARD\b"
        r"|\bPOWER BOARD\b"
        r"|\bDRIVER BOARD\b"
        r"|\bMOTHERBOARD\b"
        r"|\bCIRCUITO ELECTRONICO\b"
        r"|\bMODULOS? (?:DE|PARA)\b"
        r"|\bELECTRONIC MODULE\b"
        r"|\bPOWER MODULE\b"
        r"|\bCONTROL MODULE\b"
        r"|\bDISPLAY MODULE\b"
        r"|\bFUENTE DE PODER\b"
        r"|\bFUENTE DE ALIMENTACION\b"
        r"|\bPOWER SUPPLY\b"
        r"|\bSWITCHING POWER SUPPLY\b"
        r"|\bTRANSFORMADORES?\b"
        r"|\bTRANSFORMER\b"
        r"|\bMOTORES?\b"
        r"|\bMOTOR ASSEMBLY\b"
        r"|\bMOTOR DRIVE\b"
        r"|\bBOMBAS?\b"
        r"|\bPUMPS?\b"
        r"|\bVALVULAS?\b"
        r"|\bVALVES?\b"
        r"|\bTURBINAS?\b"
        r"|\bTURBINES?\b"
        r"|\bRODAMIENTOS?\b"
        r"|\bBEARINGS?\b"
        r"|\bENGRANAJES?\b"
        r"|\bGEARS?\b"
        r"|\bCORREAS?\b"
        r"|\bBELTS?\b"
        r"|\bPOLEAS?\b"
        r"|\bPULLEYS?\b"
        r"|\bRODILLOS?\b"
        r"|\bROLLERS?\b"
        r"|\bVENTILADORES? DE REFRIGERACION\b"
        r"|\bCOOLING FAN\b"
        r"|\bHEAT SINK\b"
        r"|\bDISIPADOR(?:ES)?\b"
        r"|\bFUSIBLES?\b"
        r"|\bFUSES?\b"
        r"|\bRELAYS?\b"
        r"|\bRELES?\b"
        r"|\bINTERRUPTORES?\b"
        r"|\bSWITCHES?\b"
        r"|\bSENSORES? INTERNOS?\b"
        r"|\bINTERNAL SENSORS?\b"
        r"|\bTUBOS? DE RAYOS? X\b"
        r"|\bX[- ]?RAY TUBES?\b"
        r"|\bTUBO RADIOGENO\b"
        r"|\bGENERADOR(?:ES)? DE RAYOS? X\b"
        r"|\bX[- ]?RAY GENERATOR\b"
        r"|\bCOLIMADOR(?:ES)?\b"
        r"|\bCOLLIMATORS?\b"
        r"|\bDETECTOR(?:ES)? DE RAYOS? X\b"
        r"|\bX[- ]?RAY DETECTORS?\b"
        r"|\bFLAT PANEL DETECTOR\b"
        r"|\bDETECTOR PANEL\b"
        r"|\bTRANSDUCTOR(?:ES)? DAÑAD[OA]S?\b"
    ),

    # ============================================================
    # 2. CONSUMIBLES
    # Elementos de utilización periódica o desechables.
    # ============================================================
    "Consumible": (
        r"\bCONSUMIBLE(?:S)?\b"
        r"|\bCONSUMABLES?\b"
        r"|\bELECTRODOS?\b"
        r"|\bELECTRODES?\b"
        r"|\bPARCHES?\b"
        r"|\bPATCHES?\b"
        r"|\bGEL(?:ES)?\b"
        r"|\bULTRASOUND GEL\b"
        r"|\bACOUSTIC GEL\b"
        r"|\bPAPEL(?:ES)?\b"
        r"|\bTERMOPAPEL\b"
        r"|\bTHERMAL PAPER\b"
        r"|\bRECORDING PAPER\b"
        r"|\bPRINTING PAPER\b"
        r"|\bROLLOS? DE PAPEL\b"
        r"|\bPAPER ROLLS?\b"
        r"|\bFILTROS?\b"
        r"|\bFILTERS?\b"
        r"|\bFILTROS? DE AIRE\b"
        r"|\bAIR FILTERS?\b"
        r"|\bCIRCUITOS? RESPIRATORIOS?\b"
        r"|\bCIRCUITOS? DE PACIENTE\b"
        r"|\bBREATHING CIRCUITS?\b"
        r"|\bPATIENT CIRCUITS?\b"
        r"|\bVENTILATOR CIRCUITS?\b"
        r"|\bTUBOS? ENDOTRAQUEALES?\b"
        r"|\bENDOTRACHEAL TUBES?\b"
        r"|\bTUBOS? DE RESPIRACION\b"
        r"|\bBREATHING TUBES?\b"
        r"|\bDESECHABLES?\b"
        r"|\bDESCARTABLES?\b"
        r"|\bDISPOSABLES?\b"
        r"|\bSINGLE USE\b"
        r"|\bUSO UNICO\b"
        r"|\bCANULAS?\b"
        r"|\bCANNULAS?\b"
        r"|\bMASCARILLAS?\b"
        r"|\bFACE MASKS?\b"
        r"|\bRESPIRATORY MASKS?\b"
        r"|\bPLACAS? RADIOGRAFICAS?\b"
        r"|\bRADIOGRAPHIC FILMS?\b"
        r"|\bX[- ]?RAY FILMS?\b"
        r"|\bCHASIS RADIOGRAFICO\b"
        r"|\bRADIOGRAPHIC CASSETTES?\b"
        r"|\bDELANTAL(?:ES)?\b"
        r"|\bPROTECTIVE APRONS?\b"
        r"|\bFORROS?\b"
        r"|\bCOVERS?\b"
        r"|\bCOVERALLS?\b"
        r"|\bGUANTES?\b"
        r"|\bGLOVES?\b"
        r"|\bBOLSAS? DE RECOLECCION\b"
        r"|\bCOLLECTION BAGS?\b"
        r"|\bBOLSAS? DE RESIDUOS\b"
        r"|\bWASTE BAGS?\b"
        r"|\bPULMONES? DE PRUEBA\b"
        r"|\bTEST LUNGS?\b"
        r"|\bFANTOMAS? DE CALIBRACION\b"
        r"|\bCALIBRATION PHANTOMS?\b"
        r"|\bDOSIMETROS? DESECHABLES?\b"
    ),

    # ============================================================
    # 3. ACCESORIOS
    # Elementos complementarios de funcionamiento o uso.
    # ============================================================
    "Accesorio": (
        r"\bACCESORIOS?\b"
        r"|\bACCESSORIES\b"
        r"|\bCABLES?\b"
        r"|\bCABLE ASSEMBL(?:Y|IES)\b"
        r"|\bPOWER CABLES?\b"
        r"|\bPATIENT CABLES?\b"
        r"|\bLEAD CABLES?\b"
        r"|\bTRANSDUCTOR(?:ES)?\b"
        r"|\bTRANSDUCERS?\b"
        r"|\bSONDAS?\b"
        r"|\bPROBES?\b"
        r"|\bULTRASOUND PROBES?\b"
        r"|\bLINEAR PROBES?\b"
        r"|\bCONVEX PROBES?\b"
        r"|\bPHASED ARRAY PROBES?\b"
        r"|\bENDOCAVITARY PROBES?\b"
        r"|\bBATERIAS?\b"
        r"|\bBATTERIES\b"
        r"|\bBATTERY PACKS?\b"
        r"|\bADAPTADOR(?:ES)?\b"
        r"|\bADAPTERS?\b"
        r"|\bPOWER ADAPTERS?\b"
        r"|\bSOPORTES?\b"
        r"|\bSUPPORTS?\b"
        r"|\bMOUNTS?\b"
        r"|\bBRACKETS?\b"
        r"|\bBRAZOS? ARTICULADOS?\b"
        r"|\bARTICULATING ARMS?\b"
        r"|\bCARROS?\b"
        r"|\bCARRITOS?\b"
        r"|\bTROLLEYS?\b"
        r"|\bTROLLEYS?\b"
        r"|\bCARTS?\b"
        r"|\bMOBILE STANDS?\b"
        r"|\bMALETIN(?:ES)?\b"
        r"|\bESTUCHES?\b"
        r"|\bCARRYING CASES?\b"
        r"|\bTRANSPORT CASES?\b"
        r"|\bMASCARAS?\b"
        r"|\bMASKS?\b"
        r"|\bHUMIDIFICADOR(?:ES)?\b"
        r"|\bHUMIDIFIERS?\b"
        r"|\bCARGADOR(?:ES)?\b"
        r"|\bCHARGERS?\b"
        r"|\bIMPRESORAS?\b"
        r"|\bPRINTERS?\b"
        r"|\bMONITORES? AUXILIARES?\b"
        r"|\bEXTERNAL MONITORS?\b"
        r"|\bPANTALLAS? AUXILIARES?\b"
        r"|\bDISPLAYS?\b"
        r"|\bTECLADOS?\b"
        r"|\bKEYBOARDS?\b"
        r"|\bPEDALES?\b"
        r"|\bFOOT SWITCHES?\b"
        r"|\bFOOT PEDALS?\b"
        r"|\bCONTROLES? REMOTOS?\b"
        r"|\bREMOTE CONTROLS?\b"
        r"|\bMANDOS? A DISTANCIA\b"
        r"|\bCOLCHON(?:ES)?\b"
        r"|\bMATTRESSES?\b"
        r"|\bCOLCHONETAS?\b"
        r"|\bPADS?\b"
        r"|\bPOSICIONADOR(?:ES)?\b"
        r"|\bPOSITIONERS?\b"
        r"|\bSENSORES?\b"
        r"|\bSENSORS?\b"
        r"|\bDETECTORES? AUXILIARES?\b"
        r"|\bAUXILIARY DETECTORS?\b"
        r"|\bTRANSMISORES?\b"
        r"|\bTRANSMITTERS?\b"
        r"|\bRECEPTORES?\b"
        r"|\bRECEIVERS?\b"
        r"|\bKIT DE MONTAJE\b"
        r"|\bMOUNTING KITS?\b"
        r"|\bACCESORIOS? DE MONTAJE\b"
        r"|\bMOUNTING ACCESSORIES\b"
    ),

    # ============================================================
    # 4. COMPONENTES
    # Partes estructurales, mecánicas y funcionales.
    # ============================================================
    "Componente": (
        r"\bCOMPONENTES?\b"
        r"|\bCOMPONENTS?\b"
        r"|\bASSEMBL(?:Y|IES)\b"
        r"|\bSUBASSEMBL(?:Y|IES)\b"
        r"|\bCONJUNTOS?\b"
        r"|\bUNIDADES? FUNCIONALES?\b"
        r"|\bFUNCTIONAL UNITS?\b"
        r"|\bESTRUCTURAS?\b"
        r"|\bSTRUCTURES?\b"
        r"|\bBASTIDORES?\b"
        r"|\bFRAMES?\b"
        r"|\bCHASIS\b"
        r"|\bCHASSIS\b"
        r"|\bCARCASAS?\b"
        r"|\bHOUSINGS?\b"
        r"|\bENCLOSURES?\b"
        r"|\bCUBIERTAS?\b"
        r"|\bCOVERS?\b"
        r"|\bPANELES?\b"
        r"|\bPANELS?\b"
        r"|\bBRAZOS?\b"
        r"|\bARMS?\b"
        r"|\bCOLUMNAS?\b"
        r"|\bCOLUMNS?\b"
        r"|\bBASES?\b"
        r"|\bBASE ASSEMBLY\b"
        r"|\bPLATAFORMAS?\b"
        r"|\bPLATFORMS?\b"
        r"|\bMECANISMOS?\b"
        r"|\bMECHANISMS?\b"
        r"|\bSISTEMAS? DE MOVIMIENTO\b"
        r"|\bMOTION SYSTEMS?\b"
        r"|\bSISTEMAS? DE POSICIONAMIENTO\b"
        r"|\bPOSITIONING SYSTEMS?\b"
        r"|\bRODILLERAS?\b"
        r"|\bARM RESTS?\b"
        r"|\bAPOYABRAZOS?\b"
        r"|\bAPOYA CABEZAS?\b"
        r"|\bHEAD RESTS?\b"
        r"|\bCOLCHONES? DE MESA\b"
        r"|\bTABLE PADS?\b"
        r"|\bBRAZOS? CEFALICOS?\b"
        r"|\bCEILING SUSPENSIONS?\b"
        r"|\bSUSPENSIONES? DE TECHO\b"
        r"|\bGANTRIES\b"
        r"|\bGANTRY\b"
        r"|\bACELERADOR(?:ES)? DE PARTICULAS\b"
        r"|\bCOLIMADOR MULTILAMINAR\b"
        r"|\bMULTILEAF COLLIMATOR\b"
        r"|\bMLC\b"
        r"|\bDETECTOR(?:ES)? PLANOS?\b"
        r"|\bFLAT PANEL\b"
        r"|\bGENERADOR(?:ES)?\b"
        r"|\bGENERATORS?\b"
        r"|\bTUBOS? RADIOGENOS?\b"
        r"|\bRADIATION TUBES?\b"
        r"|\bMODULOS? DE CONTROL\b"
        r"|\bCONTROL MODULES?\b"
        r"|\bSISTEMAS? DE REFRIGERACION\b"
        r"|\bCOOLING SYSTEMS?\b"
        r"|\bINTERCAMBIADORES? DE CALOR\b"
        r"|\bHEAT EXCHANGERS?\b"
    ),

    # ============================================================
    # 5. SERVICIOS
    # Servicios que no representan necesariamente mercancías.
    # ============================================================
    "Servicio": (
        r"\bSERVICIOS?\b"
        r"|\bSERVICES?\b"
        r"|\bMANTENIMIENTO\b"
        r"|\bMAINTENANCE\b"
        r"|\bMANTENIMIENTO PREVENTIVO\b"
        r"|\bPREVENTIVE MAINTENANCE\b"
        r"|\bMANTENIMIENTO CORRECTIVO\b"
        r"|\bCORRECTIVE MAINTENANCE\b"
        r"|\bREPARACION(?:ES)?\b"
        r"|\bREPAIRS?\b"
        r"|\bINSTALACION(?:ES)?\b"
        r"|\bINSTALLATION\b"
        r"|\bPUESTA EN FUNCIONAMIENTO\b"
        r"|\bCOMMISSIONING\b"
        r"|\bCALIBRACION(?:ES)?\b"
        r"|\bCALIBRATION\b"
        r"|\bVALIDACION(?:ES)?\b"
        r"|\bVALIDATION\b"
        r"|\bCAPACITACION(?:ES)?\b"
        r"|\bTRAINING\b"
        r"|\bFORMACION TECNICA\b"
        r"|\bTECHNICAL SUPPORT\b"
        r"|\bSOPORTE TECNICO\b"
        r"|\bASISTENCIA TECNICA\b"
        r"|\bTECHNICAL ASSISTANCE\b"
        r"|\bACTUALIZACION DE SOFTWARE\b"
        r"|\bSOFTWARE UPDATE\b"
        r"|\bLICENCIA DE SOFTWARE\b"
        r"|\bSOFTWARE LICENSE\b"
        r"|\bCONSULTORIA\b"
        r"|\bCONSULTING\b"
    ),
}

SEPARADOR_COMPONENTES = r"\b(?:CON|INCLUYE|INCLUYENDO|INCL|WITH|INCLUDING)\b"
UNIDAD_INDIVIDUAL = re.compile(r"^(U|UN|UND|UNID|UNIDAD|UNIDADES|UNIDAD\(ES\)|NIU|PZ|PZA|PIEZA|PIEZAS|EA|UNIT|UNITS|PC|PCS)\.?$")
TOLERANCIA_PESO_FICHA = 0.25
VENTANA_MARCA_MODELO = 40          # caracteres máx. entre marca y modelo en la descripción
FORMATO_NUMERICO = "auto"          # "auto" | "coma" (1.234,5) | "punto" (1,234.5)

# ---------------------------------------------------------------- columnas de las bases de importación
# Se detectan por alias (sin tildes/espacios). Si su base usa otros nombres, fíjelos en COL_MANUAL.
ALIAS_COL = {
    "subpartida":  ["SUBPARTIDA", "POSICIONARANCELARIA", "PARTIDAARANCELARIA", "SUBPARTIDAARANCELARIA", "NANDINA"],
    "descripcion": ["DESCRIPCIONMERCANCIA", "DESCRIPCIONDELAMERCANCIA", "DESCMERCANCIA", "DESCRIPCION"],
    "cantidad":    ["CANTIDAD", "CANTIDADCOMERCIAL", "CANTIDADUNIDADCOMERCIAL"],
    "unidad":      ["UNIDADCOMERCIAL", "UNIDADMEDIDA", "UNIDAD"],
    "peso_neto":   ["PESONETO", "PESONETOKG", "PESONETOKILOS"],
    "peso_bruto":  ["PESOBRUTO", "PESOBRUTOKG"],
    "fob":         ["VALORFOBUSD", "VALORFOB", "FOBUSD", "VALORFOBDOLARES"],
    "pais_origen": ["PAISORIGEN", "PAISDEORIGEN", "PAISPROCEDENCIA", "PAISDEPROCEDENCIA"],
    "importador":  ["IMPORTADOR", "RAZONSOCIALIMPORTADOR", "NOMBREIMPORTADOR", "RAZONSOCIAL"],
    "nit":         ["NITIMPORTADOR", "NIT", "IDIMPORTADOR"],
    "fecha":       ["FECHA", "FECHAREGISTRO", "FECHADECLARACION", "FECHAIMPORTACION", "FECHALEVANTE"],
    "anio":        ["ANO", "ANIO", "YEAR"],
    "mes":         ["MES", "MONTH"],
}
# Correspondencia explícita con los registros DIAN suministrados.
COL_MANUAL = {
    "subpartida": "SUBPARTIDA_ARANCELARIA",
    "descripcion": "DESCRIPCION_MERCANCIA",
    "cantidad": "CANTIDAD",
    "unidad": "UNIDAD_COMERCIAL",
    "peso_neto": "PESO_NETO",
    "peso_bruto": "PESO_BRUTO",
    "fob": "VALOR_FOB_USD",
    "pais_origen": "PAIS_ORIGEN",
    "importador": "NOMBRE_IMPORTADOR",
    "nit": "NIT_IMPORTADOR",
    "fecha": "FECHA_PRESENTACION",
}
# Clasificación estricta: los registros de otras subpartidas van a auditoría.
# No se interpretan palabras genéricas como prueba de uso biomédico.
PATRONES_EXCLUSION_INDUSTRIAL = {
    "ECOGRAFO": r"\b(?:LAVADORA|LAVADO|LIMPIEZA|SOLDADURA|SOLDAR|CORTE|CORTADORA|COSER|COSTURA|MAQUINA HERRAMIENTA|INDUSTRIAL)\b.{0,90}\bULTRASONID|\bULTRASONID.{0,90}\b(?:LAVADO|LIMPIEZA|SOLDADURA|CORTE|COSER|COSTURA|INDUSTRIAL)\b",
    "RAYOS_X": r"\b(?:INSPECCION DE EQUIPAJE|ESCANER DE EQUIPAJE|CONTROL DE SEGURIDAD|RAPISCAN|CARGA AEREA)\b",
}


# ---------------------------------------------------------------- SECOP (datos.gov.co)

SECOP_PROCESOS = "p6dx-8zbt"    # SECOP II - Procesos de contratación
SECOP_CONTRATOS = "jbjy-vk9h"   # SECOP II - Contratos electrónicos

SECOP_KW = {

    "ECOGRAFO": [
        # Denominaciones generales
        "ecografo",
        "ecografos",
        "equipo de ecografia",
        "equipos de ecografia",
        "sistema de ecografia",
        "ultrasonido diagnostico",
        "ultrasonografia",
        "ultrasonido medico",
        "sistema de ultrasonido",
        "equipo de ultrasonido",
        "diagnostico por ultrasonido",
        "doppler",
        "doppler color",
        "ultrasonido doppler",
        "ecografo doppler",

        # Denominaciones en inglés
        "ultrasound system",
        "diagnostic ultrasound",
        "ultrasound scanner",
        "ultrasonic diagnostic equipment",
        "sonography system",

        # Marcas y modelos
        "voluson",
        "logiq",
        "acuson",
        "epiq",
        "affiniti",
        "rs85",
        "hs70",
        "resona",
        "mindray ultrasound",
    ],

    "ACELERADOR_LINEAL": [
        # Denominaciones generales
        "acelerador lineal",
        "aceleradores lineales",
        "acelerador para radioterapia",
        "acelerador de particulas para radioterapia",
        "sistema de radioterapia",
        "equipo de radioterapia",
        "radioterapia externa",
        "terapia de radiacion",
        "terapia de haz externo",
        "radiacion oncologica",
        "radioterapia conformacional",
        "radioterapia de intensidad modulada",
        "imrt",
        "vmAT",
        "radiocirugia",

        # Denominaciones en inglés
        "linear accelerator",
        "medical linac",
        "linac",
        "radiotherapy system",
        "radiation therapy system",
        "external beam radiation therapy",

        # Marcas y modelos
        "truebeam",
        "halcyon",
        "versa hd",
        "elekta infinity",
        "elekta synergy",
        "elekta evo",
        "tomotherapy",
        "radixact",
    ],

    "MESA_CIRUGIA": [
        # Denominaciones generales
        "mesa de cirugia",
        "mesa quirurgica",
        "mesas quirurgicas",
        "mesa de quirofano",
        "mesa para quirofano",
        "mesa de operaciones",
        "mesa operatoria",
        "mesa para operaciones",
        "mesa de intervenciones",
        "mesa electrohidraulica",
        "mesa electromecanica",
        "mesa hidraulica quirurgica",
        "mesa quirurgica movil",
        "mesa quirurgica radiotransparente",
        "mesa de cirugia universal",

        # Denominaciones en inglés
        "surgical table",
        "operating table",
        "operation table",
        "electrohydraulic operating table",
        "electromechanical surgical table",
        "radiolucent surgical table",

        # Marcas y modelos
        "alphamaxx",
        "maquet magnus",
        "trusystem",
        "hybase",
        "steris 5085",
        "steris 4085",
    ],

    "RAYOS_X": [
        # Radiografía convencional
        "rayos x",
        "equipo de rayos x",
        "sistema de rayos x",
        "equipo de radiografia",
        "sistema de radiografia",
        "radiografia digital",
        "radiologia digital",
        "radiografia convencional",
        "equipo radiologico",
        "sistema radiologico",
        "sala de rayos x",
        "sala de radiologia",
        "radiografia computarizada",
        "radiografia directa",
        "detector digital de radiografia",
        "sistema de radiografia digital",

        # Denominaciones en inglés
        "digital radiography",
        "digital x ray system",
        "digital radiographic system",
        "fixed x ray system",
        "stationary x ray",
        "radiographic system",
        "computed radiography",
        "dr system",
        "cr system",

        # Marcas y modelos
        "ysio",
        "ysio max",
        "optima xr",
        "drx evolution",
        "digitaldiagnost",
        "luminos drf",
    ],

    "VENTILADOR": [
        # Denominaciones generales
        "ventilador mecanico",
        "ventiladores mecanicos",
        "ventilador pulmonar",
        "ventiladores pulmonares",
        "ventilador de cuidados intensivos",
        "ventilador de uci",
        "ventilacion mecanica",
        "equipo de ventilacion mecanica",
        "sistema de ventilacion mecanica",
        "respirador artificial",
        "respirador mecanico",
        "equipo de soporte ventilatorio",
        "soporte ventilatorio",
        "ventilacion invasiva",
        "ventilacion no invasiva",
        "ventilador neonatal",
        "ventilador pediatrico",

        # Denominaciones en inglés
        "mechanical ventilator",
        "mechanical ventilation",
        "icu ventilator",
        "intensive care ventilator",
        "critical care ventilator",
        "pulmonary ventilator",
        "ventilator system",
        "respiratory ventilator",

        # Marcas y modelos
        "puritan bennett 980",
        "puritan bennett 840",
        "evita v500",
        "evita v800",
        "servo u",
        "servo i",
        "servo air",
        "hamilton c6",
        "hamilton g5",
        "hamilton s1",
        "sv800",
        "carescape r860",
    ],
}


In [ ]:
%%writefile /content/bioprice_bi/bioprice/clasificacion.py
"""Objetivo 1: identificar y depurar registros de importación por subpartida + descripción, con evidencia explícita."""
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

from . import config as C


# ------------------------------------------------------------------ utilidades de texto
def norm(texto):
    if texto is None or (not isinstance(texto, str) and pd.isna(texto)):
        return ""
    t = unicodedata.normalize("NFKD", str(texto))
    return t.encode("ascii", "ignore").decode("ascii").upper()


def compacto(texto):
    return re.sub(r"[^A-Z0-9]", "", norm(texto))


def a_numero(valor, formato=None):
    formato = formato or C.FORMATO_NUMERICO
    if valor is None or (not isinstance(valor, str) and pd.isna(valor)):
        return np.nan
    if isinstance(valor, (int, float, np.number)):
        return float(valor)
    t = re.sub(r"\s", "", str(valor))
    if not t:
        return np.nan
    if formato == "coma":
        t = t.replace(".", "").replace(",", ".")
    elif formato == "punto":
        t = t.replace(",", "")
    elif "," in t and "." in t:
        t = t.replace(".", "").replace(",", ".") if t.rfind(",") > t.rfind(".") else t.replace(",", "")
    elif "," in t:
        t = t.replace(",", ".")
    try:
        return float(t)
    except ValueError:
        return np.nan


# ------------------------------------------------------------------ detección de columnas
def detectar_columnas(df):
    """Devuelve {clave: nombre_real_o_None}. Respeta C.COL_MANUAL."""
    mapa = {c: compacto(c) for c in df.columns}
    out = {}
    for k, alias in C.ALIAS_COL.items():
        if k in C.COL_MANUAL:
            nombre = C.COL_MANUAL[k]
            if nombre not in df.columns:
                raise ValueError(f"Columna configurada para {k}: {nombre!r} no existe en el archivo")
            out[k] = nombre
            continue
        hallada = None
        for a in alias:                       # 1.º exacta, 2.º parcial
            hallada = next((c for c, n in mapa.items() if n == a), None)
            if hallada:
                break
        if not hallada:
            for a in alias:
                hallada = next((c for c, n in mapa.items() if len(a) >= 5 and a in n), None)
                if hallada:
                    break
        out[k] = hallada
    return out


def normalizar_subpartida(valor):
    """Preserva dígitos de la subpartida; admite números exportados como 9018120000.0."""
    if pd.isna(valor):
        return ""
    t = str(valor).strip()
    if re.fullmatch(r"\d+\.0+", t):
        t = t.split(".")[0]
    return re.sub(r"\D", "", t)


def convertir_fechas_dian(serie):
    """FECHA_PRESENTACION: YYYYMMDD, ISO o DD/MM/YYYY sin confundir enteros con epoch."""
    texto = serie.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    ocho = texto.str.fullmatch(r"\d{8}").fillna(False)
    f = pd.Series(pd.NaT, index=serie.index, dtype="datetime64[ns]")
    f.loc[ocho] = pd.to_datetime(texto.loc[ocho], format="%Y%m%d", errors="coerce")
    resto = ~ocho
    iso = texto.loc[resto].str.match(r"^\d{4}-\d{2}-\d{2}").fillna(False)
    f.loc[texto.loc[resto].index[iso]] = pd.to_datetime(texto.loc[resto][iso], errors="coerce")
    f.loc[texto.loc[resto].index[~iso]] = pd.to_datetime(texto.loc[resto][~iso], dayfirst=True, errors="coerce")
    return f


def estandarizar(df):
    """Copia con columnas estándar (_subpartida, _descripcion, ...) SIN tocar las originales."""
    col = detectar_columnas(df)
    faltan = [k for k in ("subpartida", "descripcion") if not col[k]]
    if faltan:
        raise ValueError(f"Faltan columnas obligatorias {faltan}. Columnas disponibles: {list(df.columns)[:40]}")
    out = df.copy()
    out["_subpartida"] = df[col["subpartida"]].map(normalizar_subpartida).str[:6]
    out["_descripcion"] = df[col["descripcion"]].map(norm)
    for k in ("cantidad", "peso_neto", "peso_bruto", "fob"):
        out["_" + k] = df[col[k]].map(a_numero) if col[k] else np.nan
    out["_unidad"] = df[col["unidad"]].map(norm).str.strip() if col["unidad"] else ""
    out["_pais"] = df[col["pais_origen"]].map(norm).str.strip() if col["pais_origen"] else ""
    out["_importador"] = df[col["importador"]].map(norm).str.strip() if col["importador"] else ""
    out["_nit"] = df[col["nit"]].fillna("").astype(str).str.replace(r"\D", "", regex=True) if col["nit"] else ""
    if col["fecha"]:
        f = convertir_fechas_dian(df[col["fecha"]])
    elif col["anio"]:
        a = pd.to_numeric(df[col["anio"]], errors="coerce")
        m = pd.to_numeric(df[col["mes"]], errors="coerce").fillna(1) if col["mes"] else 1
        f = pd.to_datetime(dict(year=a, month=m, day=1), errors="coerce")
    else:
        f = pd.Series(pd.NaT, index=df.index)
    out["FECHA"] = f
    out["ANIO"] = f.dt.year
    out["PERIODO_MES"] = f.dt.to_period("M").astype(str).where(f.notna(), None)
    return out, col


PATRON_GLOBAL = "|".join(f"(?:{v['kw']})" for v in C.EQUIPOS.values())


def es_candidato(est):
    return est["_subpartida"].isin(C.SUB6_A_EQUIPO) | est["_descripcion"].str.contains(PATRON_GLOBAL, regex=True, na=False)


# ------------------------------------------------------------------ equipo / categoría / peso
def asignar_equipo(desc_n, sub6):
    por_kw = [k for k, v in C.EQUIPOS.items() if re.search(v["kw"], desc_n)]
    por_sub = C.SUB6_A_EQUIPO.get(sub6)
    if por_kw:
        equipo = por_sub if (len(por_kw) > 1 and por_sub in por_kw) else por_kw[0]
        return equipo, "descripcion", (por_kw if len(por_kw) > 1 else [])
    return por_sub, "subpartida", []


def clasificar_categoria(desc_n, equipo, origen):
    """Accesorio/consumible/repuesto solo si el término aparece ANTES del nombre del equipo y antes de «CON/INCLUYE»."""
    cabeza = re.split(C.SEPARADOR_COMPONENTES, desc_n, maxsplit=1)[0]
    hits = []
    for prio, (cat, pat) in enumerate(C.PATRONES_CATEGORIA.items()):
        m = re.search(pat, cabeza)
        if m:
            hits.append((m.start(), prio, cat, m.group(0)))
    hits.sort()
    if origen == "descripcion":
        m_eq = re.search(C.EQUIPOS[equipo]["kw"], cabeza)
        pos_eq = m_eq.start() if m_eq else 10 ** 6
        if hits and hits[0][0] < pos_eq:
            return hits[0][2], hits[0][3], None
        return "Equipo principal completo", None, (hits[0][3] if hits else None)
    if hits:
        return hits[0][2], hits[0][3], None
    return "Indeterminado", None, None


def peso_unitario(unidad, cantidad, peso_neto):
    if not unidad:
        return np.nan, "Unidad comercial ausente."
    if not C.UNIDAD_INDIVIDUAL.match(unidad):
        return np.nan, f"Unidad comercial '{unidad}' no representa unidades individuales."
    if pd.isna(cantidad) or cantidad <= 0:
        return np.nan, "Cantidad ausente o no positiva."
    if pd.isna(peso_neto) or peso_neto <= 0:
        return np.nan, "Peso neto ausente o no positivo."
    return peso_neto / cantidad, None


# ------------------------------------------------------------------ fichas técnicas (su Excel)
COLUMNAS_FICHA = ["EQUIPO", "MARCA", "MODELO", "PESO_MIN_KG", "PESO_MAX_KG", "URL_FICHA", "CONDICION_PESO", "FECHA_CONSULTA"]


def cargar_fichas(ruta=None, hoja=None):
    """Lee SU Excel (columnas EQUIPO, MARCA, MODELO, PESO_MIN_KG, PESO_MAX_KG, URL_FICHA, CONDICION_PESO, FECHA_CONSULTA).
    Descarta filas sin URL http, sin pesos o con EQUIPO fuera de las 5 familias (y lo informa)."""
    ruta = Path(ruta or C.ARCHIVO_FICHAS)
    df = pd.read_excel(ruta, sheet_name=C.HOJA_FICHAS if hoja is None else hoja, dtype=str)
    df.columns = [str(c).strip().upper() for c in df.columns]
    faltan = [c for c in COLUMNAS_FICHA if c not in df.columns]
    if faltan:
        raise ValueError(f"Al Excel de fichas le faltan columnas: {faltan}")
    return validar_fichas(df)


def validar_fichas(df):
    df = df.copy()
    df["EQUIPO"] = df["EQUIPO"].map(lambda x: re.sub(r"[^A-Z0-9]+", "_", norm(x)).strip("_"))
    df["EQUIPO"] = df["EQUIPO"].replace({"ECOGRAFOS": "ECOGRAFO", "RAYOS_X": "RAYOS_X", "RAYOSX": "RAYOS_X",
                                         "VENTILADOR_MECANICO": "VENTILADOR", "MESA_DE_CIRUGIA": "MESA_CIRUGIA",
                                         "ACELERADOR": "ACELERADOR_LINEAL"})
    for c in ("PESO_MIN_KG", "PESO_MAX_KG"):
        df[c] = df[c].map(a_numero)
    ok = (df["EQUIPO"].isin(C.EQUIPOS) & df["MARCA"].notna() & df["MODELO"].notna()
          & df["URL_FICHA"].fillna("").str.startswith("http") & df["PESO_MIN_KG"].notna() & df["PESO_MAX_KG"].notna()
          & (df["PESO_MAX_KG"] >= df["PESO_MIN_KG"]))
    if (~ok).any():
        print(f"⚠ {int((~ok).sum())} ficha(s) descartada(s): sin URL http, sin pesos, rango invertido, marca/modelo vacíos o EQUIPO no reconocido.")
    df = df[ok].reset_index(drop=True)
    df["_marca_n"], df["_modelo_n"] = df["MARCA"].map(norm), df["MODELO"].map(norm)
    return df


def _patron_flexible(texto_n, bloquear_digitos=False):
    tokens = re.findall(r"[A-Z]+|\d+", texto_n or "")
    if not tokens:
        return None
    p = r"(?<![A-Z0-9])" + r"[\s\-_/\.]*".join(re.escape(t) for t in tokens) + r"(?![A-Z0-9])"
    if bloquear_digitos and tokens[-1].isalpha():
        p += r"(?![\s\-_/]*\d)"
    return p


def coincide_marca_modelo(desc_n, marca_n, modelo_n, ventana=None):
    """True solo si marca y modelo aparecen JUNTOS (distancia <= ventana) en la descripción normalizada."""
    ventana = C.VENTANA_MARCA_MODELO if ventana is None else ventana
    c_mod = compacto(modelo_n)
    if len(c_mod) < 2:
        return False
    p_mod, p_mar = _patron_flexible(modelo_n, True), _patron_flexible(marca_n)
    if not p_mod or not p_mar:
        return False
    sd = [m.span() for m in re.finditer(p_mod, desc_n)]
    sm = [m.span() for m in re.finditer(p_mar, desc_n)]
    corto = len(c_mod) < 4
    for a, b in sd:
        rotulado = bool(re.search(r"(MODELO|MODEL|REFERENCIA|REF|MOD)\W{0,3}$", desc_n[max(0, a - 15):a]))
        for c, d in sm:
            dist = max(a - d, c - b)
            if (corto and (rotulado or dist <= 4)) or (not corto and dist <= ventana):
                return True
    return False


def buscar_ficha(desc_n, equipo, fichas):
    if fichas is None or fichas.empty:
        return None
    hall = [f for _, f in fichas[fichas["EQUIPO"] == equipo].iterrows()
            if coincide_marca_modelo(desc_n, f["_marca_n"], f["_modelo_n"])]
    return max(hall, key=lambda f: len(compacto(f["_modelo_n"]))) if hall else None


# ------------------------------------------------------------------ INVIMA (opcional, evidencia de identidad)
PATRON_REGISTRO = re.compile(r"(?<![A-Z0-9])((?:19|20)\d{2})\s*(DMA|DM|EBC)\s*-?\s*0*(\d{3,8})(?!\d)")


def claves_registro(texto_n):
    return {f"{y}{t}-{int(n)}" for y, t, n in PATRON_REGISTRO.findall(texto_n or "")}


class IndiceInvima:
    """Índice mínimo del CSV del INVIMA (solo vigentes). Verifica registro citado (A), modelo en PRODUCTO (B) y marca (C).
    'No encontrado' es NO CONCLUYENTE: el archivo solo trae registros vigentes hoy."""

    def __init__(self, df, col_registro="REGISTRO_SANITARIO", col_producto="PRODUCTO", col_marca="MARCA", col_estado=None):
        d = df.drop_duplicates(subset=[c for c in (col_registro, col_producto, col_marca) if c in df.columns]).reset_index(drop=True)
        txt = d[col_producto].map(norm) if col_producto in d.columns else pd.Series("", index=d.index)
        fam = pd.Series("", index=d.index)
        for k, cfg in C.EQUIPOS.items():
            fam = fam.mask((fam == "") & txt.str.contains(cfg["kw"], regex=True, na=False), k)
        vig = np.ones(len(d), dtype=bool)
        if col_estado and col_estado in d.columns:
            est = d[col_estado].map(norm)
            vig = (est.str.contains("VIGENTE") & ~est.str.contains("NO VIGENTE|VENCID|CANCEL|INACTIV")).to_numpy()
        self.d, self.fam, self.vig, self.txt = d, fam.to_numpy(), vig, txt
        self.marca = d[col_marca].map(norm) if col_marca in d.columns else pd.Series("", index=d.index)
        self.prod_compacto = txt.map(compacto)
        self.reg = {}
        if col_registro in d.columns:
            for i, ks in enumerate(d[col_registro].map(norm).map(claves_registro)):
                for k in ks:
                    self.reg.setdefault(k, []).append(i)

    def verificar(self, desc_n, equipo, ficha=None):
        """Devuelve (etiqueta, es_verificado_fuerte, marca_ok)."""
        a_ok = [i for k in claves_registro(desc_n) for i in self.reg.get(k, []) if self.vig[i] and self.fam[i] == equipo]
        b_ok = False
        if ficha is not None:
            mc, mk = compacto(ficha["MODELO"]), compacto(ficha["MARCA"])
            b_ok = any(self.vig[i] and self.fam[i] == equipo and mc in self.prod_compacto.iat[i] and mk[:5] in compacto(self.marca.iat[i])
                       for i in np.flatnonzero(self.fam == equipo))
        if a_ok and b_ok:
            return "Marca, modelo y registro citado verificados en INVIMA", True, True
        if b_ok:
            return "Marca y modelo verificados en INVIMA", True, True
        if a_ok:
            return "Registro citado verificado en INVIMA", True, True
        marcas = {m for m, f, v in zip(self.marca, self.fam, self.vig) if f == equipo and v and len(compacto(m)) >= 4}
        if any(compacto(m) in compacto(desc_n) for m in marcas):
            return "Marca verificada en INVIMA para esa familia (modelo no verificable)", False, True
        return "No encontrado en INVIMA vigentes (no concluyente)", False, False


# ------------------------------------------------------------------ clasificación fila a fila
def procesar_fila(r, fichas=None, invima=None):
    desc_n, sub6 = r["_descripcion"], r["_subpartida"]
    obs = []
    equipo, origen, multiples = asignar_equipo(desc_n, sub6)
    if multiples:
        obs.append("La descripción menciona varios equipos: " + ", ".join(multiples) + ".")
    if equipo is None:
        return {"EQUIPO_CLASIFICADO": "SIN ASIGNAR", "DESCARTE_TIPO": "Sin equipo asignado", "DESCARTE_DETALLE": "Ni descripción ni subpartida."}
    cfg = C.EQUIPOS[equipo]

    sub_ok = "Sí" if sub6 in cfg["sub6"] else "No"
    if sub_ok == "No":
        obs.append(f"Subpartida {sub6} no admitida para {equipo}; requiere revisión manual.")
    patron_industrial = C.PATRONES_EXCLUSION_INDUSTRIAL.get(equipo, r"(?!)")
    indicio_industrial = bool(re.search(patron_industrial, desc_n))
    if indicio_industrial:
        obs.append("Descripción compatible con uso industrial/no biomédico.")

    categoria, termino, alerta = clasificar_categoria(desc_n, equipo, origen)
    if termino:
        obs.append(f"Clasificado como {categoria.lower()} por el término '{termino}'.")
    if alerta:
        obs.append(f"Equipo completo, pero la descripción contiene '{alerta}': verificar que no sea accesorio o repuesto.")

    pu, motivo = peso_unitario(r["_unidad"], r["_cantidad"], r["_peso_neto"])
    pu_dudoso = bool(pd.notna(r["_peso_neto"]) and pd.notna(r["_peso_bruto"]) and r["_peso_neto"] > r["_peso_bruto"])
    if pu_dudoso:
        obs.append("Inconsistencia: peso neto mayor que peso bruto.")
    en_rango = None if np.isnan(pu) else cfg["peso_min"] <= pu <= cfg["peso_max"]

    ficha = buscar_ficha(desc_n, equipo, fichas)
    peso_ok, en_ficha, pmin, pmax = "Pendiente", None, np.nan, np.nan
    modelo, fuente_mod, url = "Pendiente de identificación", "", ""
    if ficha is not None:
        modelo, fuente_mod, url = f"{ficha['MARCA']} {ficha['MODELO']}", "Ficha técnica", ficha["URL_FICHA"]
        if categoria == "Equipo principal completo":
            pmin, pmax = float(ficha["PESO_MIN_KG"]), float(ficha["PESO_MAX_KG"])
            if not np.isnan(pu) and not pu_dudoso:
                en_ficha = bool(pmin * (1 - C.TOLERANCIA_PESO_FICHA) <= pu <= pmax * (1 + C.TOLERANCIA_PESO_FICHA))
                peso_ok = "Sí" if en_ficha else "No"
    else:
        obs.append("Sin ficha técnica coincidente (marca+modelo en la descripción): el peso no puede confirmarse.")

    verif, v_fuerte, marca_ok = ("INVIMA no cargado", False, False)
    if invima is not None:
        verif, v_fuerte, marca_ok = invima.verificar(desc_n, equipo, ficha)
        obs.append(f"INVIMA: {verif}.")

    flag = (alerta is not None) or pu_dudoso or indicio_industrial
    base_ok = categoria == "Equipo principal completo" and sub_ok == "Sí" and not flag
    if base_ok and ficha is not None and peso_ok == "Sí" and v_fuerte:
        nivel = "Alto"
    elif base_ok and peso_ok != "No" and (marca_ok or peso_ok == "Sí" or (ficha is None and en_rango is True)):
        nivel = "Medio"
    elif categoria in ("Accesorio", "Consumible", "Repuesto"):
        nivel = "Medio" if (sub_ok == "Sí" and origen == "descripcion") else "Bajo"
    else:
        nivel = "Bajo"

    d_tipo, d_det = "", ""
    if sub_ok == "No":
        d_tipo, d_det = "Subpartida incompatible", f"{sub6} no está en las subpartidas de referencia de {equipo}. Revisar manualmente antes de incluir."
    elif indicio_industrial:
        d_tipo, d_det = "Posible uso industrial", "La descripción menciona aplicaciones no biomédicas."
    elif categoria != "Equipo principal completo":
        d_tipo, d_det = "Categoría no principal", f"Clasificado como {categoria.lower()}."
    elif np.isnan(pu):
        d_tipo, d_det = "Sin peso unitario válido", motivo or ""
    elif not (en_rango or en_ficha):
        d_tipo = "Peso fuera de rango"
        d_det = f"Peso unitario {pu:.2f} kg fuera del rango genérico {cfg['peso_min']}–{cfg['peso_max']} kg" + (" y de la ficha." if ficha is not None else ".")

    return {"EQUIPO_CLASIFICADO": equipo, "CATEGORIA_REGISTRO": categoria, "PESO_UNITARIO_KG": pu,
            "PESO_MIN_FICHA_KG": pmin, "PESO_MAX_FICHA_KG": pmax, "MODELO_CANDIDATO": modelo, "FUENTE_MODELO": fuente_mod,
            "URL_FICHA": url, "COINCIDENCIA_PESO": peso_ok, "COINCIDENCIA_SUBPARTIDA": sub_ok, "VERIFICACION_INVIMA": verif,
            "NIVEL_DE_CONFIANZA": nivel, "ESTADO_IDENTIFICACION": "Excluido" if d_tipo else ("Confirmado" if nivel == "Alto" else "Candidato"),
            "OBSERVACIONES": " | ".join(obs), "DESCARTE_TIPO": d_tipo, "DESCARTE_DETALLE": d_det}


def clasificar(df_crudo, fichas=None, invima=None):
    """Devuelve (conservados, descartados, columnas_detectadas). Los datos originales no se modifican."""
    est, col = estandarizar(df_crudo)
    est = est[es_candidato(est)].copy()
    res = pd.DataFrame([procesar_fila(r, fichas, invima) for _, r in est.iterrows()], index=est.index)
    todo = pd.concat([est, res], axis=1)
    desc = todo["DESCARTE_TIPO"].fillna("") != ""
    return todo[~desc].copy(), todo[desc].copy(), col



In [ ]:

%%writefile /content/bioprice_bi/bioprice/indicadores.py
"""Objetivos 2 y 5: precio unitario FOB, variación temporal, comparación por país y actores importadores."""

import numpy as np
import pandas as pd

from . import config as C


def preparar(df, min_confianza=None, solo_principal=True):
    """Agrega PRECIO_UNIT_USD = FOB / cantidad si la unidad comercial es individual."""
    d = df.copy()

    if solo_principal and "CATEGORIA_REGISTRO" in d:
        d = d[d["CATEGORIA_REGISTRO"] == "Equipo principal completo"]

    if min_confianza:
        orden = {"Bajo": 0, "Medio": 1, "Alto": 2}
        d = d[d["NIVEL_DE_CONFIANZA"].map(orden) >= orden[min_confianza]]

    indiv = d["_unidad"].map(
        lambda u: bool(C.UNIDAD_INDIVIDUAL.match(u or ""))
    )

    ok = indiv & (d["_cantidad"] > 0) & (d["_fob"] > 0)

    d["PRECIO_UNIT_USD"] = np.where(
        ok,
        d["_fob"] / d["_cantidad"].where(d["_cantidad"] > 0),
        np.nan
    )

    return d


def _q(p):
    return lambda s: s.quantile(p)


def serie_precios(d, freq="Y", por=("EQUIPO_CLASIFICADO",), min_n=5):
    """Mediana/P25/P75 del precio unitario FOB y variación porcentual interanual."""

    x = d.dropna(subset=["PRECIO_UNIT_USD", "FECHA"]).copy()
    x["PERIODO"] = x["FECHA"].dt.to_period(freq)

    por = list(por)

    g = x.groupby(por + ["PERIODO"]).agg(
        n=("PRECIO_UNIT_USD", "size"),
        mediana=("PRECIO_UNIT_USD", "median"),
        p25=("PRECIO_UNIT_USD", _q(0.25)),
        p75=("PRECIO_UNIT_USD", _q(0.75)),
        unidades=("_cantidad", "sum"),
        valor_fob=("_fob", "sum")
    ).reset_index()

    g["precio_ponderado"] = (
        g["valor_fob"] / g["unidades"].replace(0, np.nan)
    )

    g = g.sort_values(por + ["PERIODO"])

    g["VAR_PCT_MEDIANA"] = (
        g.groupby(por)["mediana"].pct_change() * 100
    )

    g["MUESTRA_PEQUENA"] = g["n"] < min_n
    g["PERIODO"] = g["PERIODO"].astype(str)

    return g


def comparacion_pais(d, min_n=3):
    """Precio FOB por país de origen y prima/descuento frente a la mediana de la familia."""

    x = d.dropna(subset=["PRECIO_UNIT_USD"]).copy()

    g = x.groupby(["EQUIPO_CLASIFICADO", "_pais"]).agg(
        n=("PRECIO_UNIT_USD", "size"),
        mediana=("PRECIO_UNIT_USD", "median"),
        unidades=("_cantidad", "sum"),
        valor_fob=("_fob", "sum")
    ).reset_index().rename(
        columns={"_pais": "PAIS_ORIGEN"}
    )

    med = x.groupby("EQUIPO_CLASIFICADO")[
        "PRECIO_UNIT_USD"
    ].median().rename("mediana_familia")

    g = g.join(med, on="EQUIPO_CLASIFICADO")

    g["PRIMA_PCT_VS_FAMILIA"] = (
        (g["mediana"] / g["mediana_familia"] - 1) * 100
    )

    g["CUOTA_UNIDADES_PCT"] = (
        g["unidades"] /
        g.groupby("EQUIPO_CLASIFICADO")["unidades"].transform("sum") * 100
    )

    g["MUESTRA_PEQUENA"] = g["n"] < min_n

    return g.sort_values(
        ["EQUIPO_CLASIFICADO", "valor_fob"],
        ascending=[True, False]
    )


def _clave_actor(d):
    nit = d["_nit"].fillna("")
    return np.where(nit.str.len() >= 6, nit, d["_importador"])


def perfil_importadores(d):
    """Objetivo 5: actividad por importador y familia, medida mediante valor FOB."""

    x = d.dropna(subset=["FECHA"]).copy()
    x["ACTOR"] = _clave_actor(x)
    x = x[x["ACTOR"].astype(str) != ""]

    g = x.groupby(["EQUIPO_CLASIFICADO", "ACTOR"]).agg(
        IMPORTADOR=(
            "_importador",
            lambda s: s.mode().iat[0] if len(s.mode()) else ""
        ),
        registros=("_fob", "size"),
        unidades=("_cantidad", "sum"),
        valor_fob=("_fob", "sum"),
        primer_anio=("ANIO", "min"),
        ultimo_anio=("ANIO", "max"),
        anios_activos=("ANIO", "nunique"),
        precio_mediano=("PRECIO_UNIT_USD", "median")
    ).reset_index()

    g["CUOTA_VALOR_PCT"] = (
        g["valor_fob"] /
        g.groupby("EQUIPO_CLASIFICADO")["valor_fob"].transform("sum") * 100
    )

    def pendiente(sub):
        a = x[
            (x["EQUIPO_CLASIFICADO"] == sub["EQUIPO_CLASIFICADO"]) &
            (x["ACTOR"] == sub["ACTOR"])
        ]

        s = a.groupby("ANIO")["_fob"].sum()

        return (
            np.polyfit(s.index.astype(float), s.values, 1)[0]
            if len(s) >= 3 else np.nan
        )

    g["TENDENCIA_VALOR_FOB_USD_ANIO"] = g.apply(pendiente, axis=1)

    return g.sort_values(
        ["EQUIPO_CLASIFICADO", "valor_fob"],
        ascending=[True, False]
    )


def concentracion(d):
    """HHI (0–10.000) y cuota del top-5 por familia y año, sobre valor FOB."""

    x = d.dropna(subset=["ANIO"]).copy()
    x["ACTOR"] = _clave_actor(x)

    s = x.groupby(
        ["EQUIPO_CLASIFICADO", "ANIO", "ACTOR"]
    )["_fob"].sum().reset_index()

    s["cuota"] = (
        s["_fob"] /
        s.groupby(["EQUIPO_CLASIFICADO", "ANIO"])["_fob"].transform("sum") * 100
    )

    out = s.groupby(
        ["EQUIPO_CLASIFICADO", "ANIO"]
    ).agg(
        HHI=("cuota", lambda c: float((c ** 2).sum())),
        TOP5_PCT=("cuota", lambda c: float(c.nlargest(5).sum())),
        n_importadores=("ACTOR", "nunique")
    ).reset_index()

    return out


def evolucion_cuotas(d, familia, top=10):
    """Evolución de la participación de importadores según valor FOB."""

    x = d[
        d["EQUIPO_CLASIFICADO"] == familia
    ].dropna(subset=["ANIO"]).copy()

    x["ACTOR"] = _clave_actor(x)

    x["NOMBRE"] = x["_importador"].where(
        x["_importador"] != "",
        x["ACTOR"]
    )

    grandes = x.groupby("NOMBRE")["_fob"].sum().nlargest(top).index

    x["NOMBRE"] = x["NOMBRE"].where(
        x["NOMBRE"].isin(grandes),
        "OTROS"
    )

    t = x.groupby(
        ["ANIO", "NOMBRE"]
    )["_fob"].sum().reset_index()

    t["CUOTA_PCT"] = (
        t["_fob"] /
        t.groupby("ANIO")["_fob"].transform("sum") * 100
    )

    return t


In [ ]:
%%writefile /content/bioprice_bi/bioprice/ml.py
"""Machine learning (apoyo analítico, nunca sustituye la verificación documental):
1. Anomalías de precio (IsolationForest por familia; MAD robusto si hay pocos datos).
2. Precio esperado (HistGradientBoosting con validación cruzada) y razón precio real / esperado.
3. Clasificador de texto (TF-IDF + regresión logística) que SUGIERE categoría para descripciones ambiguas.
4. Segmentación de importadores (KMeans).
"""
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingRegressor, IsolationForest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

SEMILLA = 42


def marcar_anomalias(d, contaminacion=0.05, min_n=40, umbral_mad=3.5):
    d = d.copy()
    d["ES_ANOMALIA_PRECIO"], d["SCORE_ANOMALIA"], d["METODO_ANOMALIA"] = False, np.nan, ""
    for fam, idx in d.dropna(subset=["PRECIO_UNIT_USD"]).groupby("EQUIPO_CLASIFICADO").groups.items():
        s = d.loc[idx]
        lp = np.log(s["PRECIO_UNIT_USD"].clip(lower=0.01))
        if len(s) >= min_n:
            X = pd.DataFrame({"lp": lp, "lw": np.log(s["PESO_UNITARIO_KG"].clip(lower=0.01)).fillna(lp.median())})
            m = IsolationForest(contamination=contaminacion, random_state=SEMILLA).fit(X)
            d.loc[idx, "ES_ANOMALIA_PRECIO"] = m.predict(X) == -1
            d.loc[idx, "SCORE_ANOMALIA"] = -m.score_samples(X)
            d.loc[idx, "METODO_ANOMALIA"] = "IsolationForest"
        elif len(s) >= 8:
            med, mad = lp.median(), (lp - lp.median()).abs().median() or 1e-9
            z = 0.6745 * (lp - med) / mad
            d.loc[idx, "ES_ANOMALIA_PRECIO"] = z.abs() > umbral_mad
            d.loc[idx, "SCORE_ANOMALIA"] = z.abs()
            d.loc[idx, "METODO_ANOMALIA"] = "MAD robusto"
    return d


def precio_esperado(d, min_filas=60, folds=5):
    """Predicción fuera de muestra (cross_val_predict) del log-precio. Devuelve (d_con_columnas, métricas)."""
    x = d.dropna(subset=["PRECIO_UNIT_USD"]).copy()
    x = x[~x["ES_ANOMALIA_PRECIO"]] if "ES_ANOMALIA_PRECIO" in x else x
    if len(x) < min_filas:
        return d.assign(PRECIO_ESPERADO_USD=np.nan, RAZON_PRECIO_ESPERADO=np.nan), {"estado": f"Insuficiente ({len(x)} filas < {min_filas})"}
    cats = ["EQUIPO_CLASIFICADO", "_pais", "MODELO_CANDIDATO"]
    nums = ["ANIO", "PESO_UNITARIO_KG"]
    X = pd.concat([x[cats].fillna("ND"), x[nums]], axis=1)
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cats] = enc.fit_transform(X[cats])
    y = np.log(x["PRECIO_UNIT_USD"].clip(lower=0.01))
    modelo = HistGradientBoostingRegressor(random_state=SEMILLA, categorical_features=[0, 1, 2], max_iter=200, learning_rate=0.06)
    kf = KFold(n_splits=min(folds, len(x)), shuffle=True, random_state=SEMILLA)
    pred = cross_val_predict(modelo, X.assign(**{c: X[c].astype(int) for c in cats}), y, cv=kf)
    base = np.full(len(y), y.mean())
    met = {"estado": "ok", "n": int(len(x)), "R2_cv": float(r2_score(y, pred)), "MAE_log_cv": float(mean_absolute_error(y, pred)),
           "MAE_log_base_media": float(mean_absolute_error(y, base)),
           "interpretacion": "El modelo solo aporta si MAE_log_cv < MAE_log_base_media."}
    out = d.copy()
    out["PRECIO_ESPERADO_USD"], out["RAZON_PRECIO_ESPERADO"] = np.nan, np.nan
    out.loc[x.index, "PRECIO_ESPERADO_USD"] = np.exp(pred)
    out.loc[x.index, "RAZON_PRECIO_ESPERADO"] = x["PRECIO_UNIT_USD"] / np.exp(pred)
    return out, met


def clasificador_categoria(d, min_por_clase=15):
    """Aprendizaje con etiquetas DÉBILES (las reglas, confianza Alto/Medio). Reporta F1 en validación cruzada y
    devuelve (modelo|None, métricas). Sirve para SUGERIR categoría; no sobrescribe la regla."""
    t = d[d["NIVEL_DE_CONFIANZA"].isin(["Alto", "Medio"])]
    cuentas = t["CATEGORIA_REGISTRO"].value_counts()
    t = t[t["CATEGORIA_REGISTRO"].isin(cuentas[cuentas >= min_por_clase].index)]
    if t["CATEGORIA_REGISTRO"].nunique() < 2:
        return None, {"estado": "Insuficiente: se necesitan ≥2 categorías con suficientes ejemplos"}
    pipe = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                         LogisticRegression(max_iter=1000, class_weight="balanced"))
    f1 = cross_val_score(pipe, t["_descripcion"], t["CATEGORIA_REGISTRO"], cv=min(5, int(cuentas.min())), scoring="f1_macro")
    pipe.fit(t["_descripcion"], t["CATEGORIA_REGISTRO"])
    return pipe, {"estado": "ok", "n_entrenamiento": int(len(t)), "F1_macro_cv": float(f1.mean()),
                  "advertencia": "Etiquetas débiles: el F1 mide acuerdo con las reglas, no exactitud real. Valide con una muestra revisada a mano."}


def evaluar_con_muestra_manual(modelo, muestra):
    """muestra: DataFrame con _descripcion y CATEGORIA_MANUAL (revisada por una persona). Devuelve F1 macro real."""
    return float(f1_score(muestra["CATEGORIA_MANUAL"], modelo.predict(muestra["_descripcion"]), average="macro"))


def segmentar_importadores(perfil, k=4):
    p = perfil[perfil["valor_fob"] > 0].copy()
    if len(p) < k * 3:
        p["SEGMENTO"] = "Datos insuficientes"
        return p
    X = StandardScaler().fit_transform(np.c_[np.log1p(p["valor_fob"]), np.log1p(p["registros"]), p["anios_activos"]])
    p["_c"] = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit_predict(X)
    rango = p.groupby("_c")["valor_fob"].median().rank(ascending=False).astype(int)
    nombres = {1: "Grandes y recurrentes", 2: "Medianos", 3: "Pequeños", 4: "Ocasionales"}
    p["SEGMENTO"] = p["_c"].map(rango).map(lambda r: nombres.get(r, f"Grupo {r}"))
    return p.drop(columns="_c")

In [ ]:
%%writefile /content/bioprice_bi/bioprice/secop.py
"""Licitaciones/contratos públicos (SECOP II, datos.gov.co). Las columnas se descubren desde los metadatos del dataset
(no se asumen): el valor es de PROCESO/CONTRATO, no precio unitario."""
import re

import pandas as pd
import requests

from . import config as C
from .clasificacion import clasificar_categoria, norm

URL = "https://www.datos.gov.co"
# nombre estándar -> posibles nombres reales (fieldName) en cada dataset. Se usan solo los que existan.
CANDIDATOS = {
    "ID": ["id_del_proceso", "id_contrato", "referencia_del_proceso", "referencia_del_contrato"],
    "ENTIDAD": ["entidad", "nombre_entidad"],
    "NIT_ENTIDAD": ["nit_entidad"],
    "DEPARTAMENTO": ["departamento_entidad", "departamento"],
    "CIUDAD": ["ciudad_entidad", "ciudad"],
    "MODALIDAD": ["modalidad_de_contratacion"],
    "DESCRIPCION": ["descripci_n_del_procedimiento", "descripcion_del_proceso", "nombre_del_procedimiento", "objeto_del_contrato"],
    "VALOR_ESTIMADO": ["precio_base", "valor_del_contrato"],
    "VALOR_ADJUDICADO": ["valor_total_adjudicacion", "valor_pagado"],
    "PROVEEDOR": ["nombre_del_proveedor", "proveedor_adjudicado"],
    "NIT_PROVEEDOR": ["nit_del_proveedor_adjudicado", "documento_proveedor"],
    "FECHA": ["fecha_de_publicacion_del", "fecha_de_firma", "fecha_de_publicacion_fase_3"],
    "ESTADO": ["estado_del_procedimiento", "estado_contrato", "estado_de_apertura_del_proceso", "fase"],
    "URL": ["urlproceso"],
}


def columnas_dataset(dataset):
    r = requests.get(f"{URL}/api/views/{dataset}.json", timeout=60)
    r.raise_for_status()
    return [c["fieldName"] for c in r.json().get("columns", [])]


def _mapa(cols):
    return {std: next((c for c in cand if c in cols), None) for std, cand in CANDIDATOS.items()}


def descargar(dataset, familias=None, limite=5000, texto_extra=None):
    """Descarga registros cuyo texto descriptivo contiene alguna palabra clave de las familias elegidas."""
    cols = columnas_dataset(dataset)
    m = _mapa(cols)
    if not m["DESCRIPCION"]:
        raise ValueError(f"No se halló columna descriptiva en {dataset}. Columnas: {cols}")
    kws = [k for f in (familias or C.SECOP_KW) for k in C.SECOP_KW[f]] + list(texto_extra or [])
    kws = list(dict.fromkeys(norm(k).lower() for k in kws))
    campo = m["DESCRIPCION"]
    cond = " OR ".join(f"lower({campo}) like '%{k.replace(chr(39), '')}%'" for k in kws)
    filas, off = [], 0
    while off < limite:
        r = requests.get(f"{URL}/resource/{dataset}.json", timeout=120,
                         params={"$where": cond, "$limit": min(1000, limite - off), "$offset": off, "$order": ":id"})
        r.raise_for_status()
        lote = r.json()
        if not lote:
            break
        filas += lote
        off += len(lote)
        if len(lote) < 1000:
            break
    return normalizar(pd.DataFrame(filas), dataset)


def _plano(v):
    return v.get("url") if isinstance(v, dict) else v


def normalizar(df, dataset="", columnas_extra=()):
    m = _mapa(list(df.columns))
    out = pd.DataFrame({std: df[c] if c else pd.NA for std, c in m.items()}) if len(df) else pd.DataFrame(columns=list(m))
    out["URL"] = out["URL"].map(_plano)
    for c in ("VALOR_ESTIMADO", "VALOR_ADJUDICADO"):
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out["FECHA"] = pd.to_datetime(out["FECHA"], errors="coerce")
    out["ANIO"] = out["FECHA"].dt.year
    out["DATASET"] = dataset
    d_n = out["DESCRIPCION"].fillna("").map(norm)
    fam = pd.Series("", index=out.index)
    for k, cfg in C.EQUIPOS.items():
        fam = fam.mask((fam == "") & d_n.str.contains(cfg["kw"], regex=True, na=False), k)
    out["FAMILIA"] = fam
    out = out[out["FAMILIA"] != ""].copy()
    d_n = d_n.loc[out.index]
    out["CATEGORIA_LICITACION"] = [clasificar_categoria(t, f, "descripcion")[0] for t, f in zip(d_n, out["FAMILIA"])]
    out["ADVERTENCIA"] = "Valor del proceso/contrato (puede incluir varias unidades, servicios o accesorios): no es precio unitario."
    return out.reset_index(drop=True)

In [ ]:
import os, sys
os.chdir('/content/bioprice_bi')
sys.path.insert(0, '/content/bioprice_bi')
import importlib, pandas as pd, numpy as np
import plotly.express as px
from bioprice import config as C, clasificacion as CL, indicadores as I, ml, secop
for m in (C, CL, I, ml, secop): importlib.reload(m)   # útil si edita un módulo y reejecuta
print('Base:', C.BASE)

In [ ]:
# Verificación rápida de las reglas corregidas antes del procesamiento masivo.
from bioprice import clasificacion as CL, config as C
import pandas as pd
prueba = pd.DataFrame([
    {"SUBPARTIDA_ARANCELARIA":"8456200000", "DESCRIPCION_MERCANCIA":"MAQUINA QUE OPERA POR ULTRASONIDO INDUSTRIAL", "FECHA_PRESENTACION":"20210125", "CANTIDAD":"1", "UNIDAD_COMERCIAL":"UNIDAD", "PESO_NETO":"100", "PESO_BRUTO":"110", "VALOR_FOB_USD":"10000", "PAIS_ORIGEN":"ITALIA", "NOMBRE_IMPORTADOR":"DISMOTOS", "NIT_IMPORTADOR":"900050616"},
    {"SUBPARTIDA_ARANCELARIA":"9018120000", "DESCRIPCION_MERCANCIA":"ECOGRAFO PARA DIAGNOSTICO MEDICO", "FECHA_PRESENTACION":"2021-01-25", "CANTIDAD":"1", "UNIDAD_COMERCIAL":"UNIDAD", "PESO_NETO":"100", "PESO_BRUTO":"110", "VALOR_FOB_USD":"10000", "PAIS_ORIGEN":"USA", "NOMBRE_IMPORTADOR":"PRUEBA", "NIT_IMPORTADOR":"123"},
])
normalizado, cols = CL.estandarizar(prueba)
assert normalizado["ANIO"].tolist() == [2021, 2021]
assert normalizado["_subpartida"].tolist() == ["845620", "901812"]
buenos_prueba, malos_prueba, _ = CL.clasificar(prueba)
assert "DISMOTOS" not in buenos_prueba["_importador"].tolist()
assert "Subpartida incompatible" in malos_prueba["DESCARTE_TIPO"].tolist()
print("Pruebas de fecha, subpartida y exclusión de Dismotos: correctas")


## 3. Conversión de los Excel de importaciones a Parquet
Se conserva su celda original (lee `_EXTRAIDOS` y escribe `_PARQUET`, omite lo ya convertido).

In [ ]:
import os
import re
import csv
import math
import hashlib
from pathlib import Path
from datetime import date, datetime

from python_calamine_plus import load_workbook
import pyarrow as pa
import pyarrow.parquet as pq


# ============================================================
# CONFIGURACIÓN
# ============================================================

CARPETA_BASE = Path(
    "/content/drive/MyDrive/REGISTROS_IMPORTACIONES"
)

CARPETA_ENTRADA = CARPETA_BASE / "_EXTRAIDOS"
CARPETA_SALIDA = CARPETA_BASE / "_PARQUET"

ARCHIVO_MANIFIESTO = CARPETA_SALIDA / "manifiesto.csv"

FILA_ENCABEZADOS = 1
TAMANO_BLOQUE = 50_000

COMPRESION = "snappy"

EXTENSIONES = {".xlsx", ".xlsm"}


# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def limpiar_nombre(nombre):
    nombre = str(nombre).strip()
    nombre = re.sub(r'[\\/*?:"<>|]', "_", nombre)
    nombre = re.sub(r"\s+", "_", nombre)
    return nombre[:100] or "sin_nombre"


def convertir_texto(valor):

    if valor is None:
        return None

    if isinstance(valor, (datetime, date)):
        return valor.isoformat()

    if isinstance(valor, float):
        if math.isnan(valor) or math.isinf(valor):
            return None

    return str(valor).strip()


def normalizar_encabezados(fila):

    encabezados = []
    contador = {}

    for i, valor in enumerate(fila):

        nombre = convertir_texto(valor)

        if not nombre:
            nombre = f"COLUMNA_{i + 1}"

        nombre = nombre.strip()

        contador[nombre] = contador.get(nombre, 0) + 1

        if contador[nombre] > 1:
            nombre = f"{nombre}_{contador[nombre]}"

        encabezados.append(nombre)

    return encabezados


def obtener_ruta_salida(ruta_excel, indice_hoja, nombre_hoja):

    relativa = ruta_excel.relative_to(CARPETA_ENTRADA)

    carpeta_destino = CARPETA_SALIDA / relativa.parent

    carpeta_destino.mkdir(parents=True, exist_ok=True)

    nombre_archivo = (
        f"{ruta_excel.stem}"
        f"__hoja_{indice_hoja:02d}_"
        f"{limpiar_nombre(nombre_hoja)}.parquet"
    )

    return carpeta_destino / nombre_archivo


def cargar_manifiesto():

    registros = {}

    if not ARCHIVO_MANIFIESTO.exists():
        return registros

    with open(
        ARCHIVO_MANIFIESTO,
        "r",
        encoding="utf-8",
        newline=""
    ) as archivo:

        lector = csv.DictReader(archivo)

        for fila in lector:

            if fila.get("estado") == "COMPLETADO":
                clave = (
                    fila["archivo_origen"],
                    fila["hoja"]
                )

                registros[clave] = fila

    return registros


def registrar_resultado(datos):

    existe = ARCHIVO_MANIFIESTO.exists()

    with open(
        ARCHIVO_MANIFIESTO,
        "a",
        encoding="utf-8",
        newline=""
    ) as archivo:

        campos = [
            "archivo_origen",
            "hoja",
            "archivo_salida",
            "tamano_origen",
            "modificado_ns",
            "filas",
            "estado"
        ]

        escritor = csv.DictWriter(
            archivo,
            fieldnames=campos
        )

        if not existe:
            escritor.writeheader()

        escritor.writerow(datos)


# ============================================================
# CONVERSIÓN DE UNA HOJA
# ============================================================

def convertir_hoja(
    ruta_excel,
    nombre_hoja,
    indice_hoja,
    manifiesto
):

    ruta_relativa = str(
        ruta_excel.relative_to(CARPETA_ENTRADA)
    )

    ruta_parquet = obtener_ruta_salida(
        ruta_excel,
        indice_hoja,
        nombre_hoja
    )

    tamano_origen = ruta_excel.stat().st_size
    modificado_ns = ruta_excel.stat().st_mtime_ns

    clave = (ruta_relativa, nombre_hoja)

        # Si el Parquet ya existe, no volver a procesar el Excel.
    if ruta_parquet.exists():
        print(f"OMITIDO (YA EXISTE): {ruta_relativa} | {nombre_hoja}")
        return
    temporal = ruta_parquet.with_name(
        ruta_parquet.stem + ".tmp.parquet"
    )

    filas_procesadas = 0
    escritor = None

    try:

        libro = load_workbook(
            ruta_excel,
            compatibility="openpyxl",
            read_only=True,
            data_only=True
        )

        hoja = libro[nombre_hoja]

        iterador = hoja.iter_rows(
            min_row=FILA_ENCABEZADOS,
            values_only=True
        )

        encabezados_originales = next(iterador, None)

        if encabezados_originales is None:
            libro.close()
            return

        encabezados = normalizar_encabezados(
            encabezados_originales
        )

        campos = [
            pa.field(nombre, pa.string())
            for nombre in encabezados
        ]

        campos.extend([
            pa.field("__ARCHIVO_ORIGEN", pa.string()),
            pa.field("__HOJA_ORIGEN", pa.string()),
            pa.field("__FILA_EXCEL", pa.int64())
        ])

        esquema = pa.schema(campos)

        escritor = pq.ParquetWriter(
            temporal,
            esquema,
            compression=COMPRESION
        )

        bloque = []

        for numero_fila, fila in enumerate(
            iterador,
            start=FILA_ENCABEZADOS + 1
        ):

            if not any(valor is not None for valor in fila):
                continue

            registro_fila = {}

            for i, encabezado in enumerate(encabezados):

                valor = fila[i] if i < len(fila) else None

                registro_fila[encabezado] = convertir_texto(
                    valor
                )

            registro_fila["__ARCHIVO_ORIGEN"] = ruta_relativa
            registro_fila["__HOJA_ORIGEN"] = nombre_hoja
            registro_fila["__FILA_EXCEL"] = numero_fila

            bloque.append(registro_fila)

            if len(bloque) >= TAMANO_BLOQUE:

                tabla = pa.Table.from_pylist(
                    bloque,
                    schema=esquema
                )

                escritor.write_table(tabla)

                filas_procesadas += len(bloque)
                bloque.clear()

        if bloque:

            tabla = pa.Table.from_pylist(
                bloque,
                schema=esquema
            )

            escritor.write_table(tabla)

            filas_procesadas += len(bloque)

        escritor.close()
        escritor = None

        libro.close()

        if filas_procesadas == 0:

            temporal.unlink(missing_ok=True)

            print(
                f"HOJA VACÍA: {ruta_relativa} | {nombre_hoja}"
            )

            return

        os.replace(temporal, ruta_parquet)

        resultado = {
            "archivo_origen": ruta_relativa,
            "hoja": nombre_hoja,
            "archivo_salida": str(
                ruta_parquet.relative_to(CARPETA_SALIDA)
            ),
            "tamano_origen": tamano_origen,
            "modificado_ns": modificado_ns,
            "filas": filas_procesadas,
            "estado": "COMPLETADO"
        }

        registrar_resultado(resultado)
        manifiesto[clave] = resultado

        print(
            f"COMPLETADO: {ruta_relativa} | "
            f"{nombre_hoja} | "
            f"{filas_procesadas:,} filas"
        )

    except Exception as error:

        if escritor is not None:
            escritor.close()

        temporal.unlink(missing_ok=True)

        print(
            f"ERROR: {ruta_relativa} | "
            f"{nombre_hoja} | {error}"
        )

        registrar_resultado({
            "archivo_origen": ruta_relativa,
            "hoja": nombre_hoja,
            "archivo_salida": str(ruta_parquet),
            "tamano_origen": tamano_origen,
            "modificado_ns": modificado_ns,
            "filas": filas_procesadas,
            "estado": "ERROR"
        })


# ============================================================
# PROCESAMIENTO GENERAL
# ============================================================

def convertir_todos_los_excel():

    CARPETA_SALIDA.mkdir(
        parents=True,
        exist_ok=True
    )

    manifiesto = cargar_manifiesto()

    archivos = sorted(
        ruta
        for ruta in CARPETA_ENTRADA.rglob("*")
        if ruta.is_file()
        and ruta.suffix.lower() in EXTENSIONES
        and not ruta.name.startswith("~$")
    )

    print(f"Archivos Excel encontrados: {len(archivos)}")

    for indice, ruta_excel in enumerate(archivos, start=1):

        print(
            f"\n[{indice}/{len(archivos)}] "
            f"Procesando: {ruta_excel.name}"
        )

        try:

            libro = load_workbook(
                ruta_excel,
                compatibility="openpyxl",
                read_only=True,
                data_only=True
            )

            nombres_hojas = list(libro.sheetnames)

            libro.close()

            for indice_hoja, nombre_hoja in enumerate(
                nombres_hojas,
                start=1
            ):

                convertir_hoja(
                    ruta_excel,
                    nombre_hoja,
                    indice_hoja,
                    manifiesto
                )

        except Exception as error:

            print(
                f"ERROR AL ABRIR: {ruta_excel} | {error}"
            )

    print("\nPROCESO FINALIZADO")


# ============================================================
# EJECUTAR
# ============================================================

if not CARPETA_ENTRADA.exists():
    raise FileNotFoundError(
        f"No existe la carpeta: {CARPETA_ENTRADA}"
    )

convertir_todos_los_excel()

## 4. Fichas técnicas y verificación INVIMA

**Fichas:** se lee su Excel con las columnas `EQUIPO, MARCA, MODELO, PESO_MIN_KG, PESO_MAX_KG, URL_FICHA, CONDICION_PESO, FECHA_CONSULTA`. Se descartan (y se informa) filas sin URL `http`, sin pesos, con rango invertido o con `EQUIPO` fuera de las 5 familias. Valores válidos de `EQUIPO`: `ECOGRAFO, ACELERADOR_LINEAL, MESA_CIRUGIA, RAYOS_X, VENTILADOR`.

**INVIMA:** si el `.xlsx` está en la carpeta base, se convierte a CSV aquí mismo.

In [ ]:
# --- Ajuste el nombre real de su Excel de fichas (en la carpeta base)
C.ARCHIVO_FICHAS = C.BASE / 'fichas_tecnicas.xlsx'
C.HOJA_FICHAS = 0
fichas = CL.cargar_fichas()
print('Fichas válidas:', len(fichas))
display(fichas.drop(columns=['_marca_n', '_modelo_n']).groupby('EQUIPO').size().rename('fichas').to_frame())
display(fichas.drop(columns=['_marca_n', '_modelo_n']).head(20))

In [ ]:
import pandas as pd
import re
from bioprice import config as C
from bioprice.clasificacion import norm, a_numero

# Vamos a cargar el Excel sin aplicar filtros para inspeccionar por qué se descartan las filas
df_crudo = pd.read_excel(C.ARCHIVO_FICHAS, sheet_name=C.HOJA_FICHAS, dtype=str)
df_crudo.columns = [str(c).strip().upper() for c in df_crudo.columns]

print(f"Total de filas leídas en bruto del Excel: {len(df_crudo)}")

# Analicemos paso a paso cada fila para ver cuál regla de validación falla
reporte_descartes = []

for idx, row in df_crudo.iterrows():
    motivos = []

    # 1. Validar Equipo
    eq_original = row.get("EQUIPO")
    eq_norm = re.sub(r"[^A-Z0-9]+", "_", norm(eq_original)).strip("_") if pd.notna(eq_original) else ""
    eq_mapeado = eq_norm
    if eq_norm == "ECOGRAFOS": eq_mapeado = "ECOGRAFO"
    elif eq_norm in ("RAYOS_X", "RAYOSX"): eq_mapeado = "RAYOS_X"
    elif eq_norm == "VENTILADOR_MECANICO": eq_mapeado = "VENTILADOR"
    elif eq_norm == "MESA_DE_CIRUGIA": eq_mapeado = "MESA_CIRUGIA"
    elif eq_norm == "ACELERADOR": eq_mapeado = "ACELERADOR_LINEAL"

    if eq_mapeado not in C.EQUIPOS:
        motivos.append(f"EQUIPO no reconocido ('{eq_original}')")

    # 2. Validar Marca y Modelo
    if pd.isna(row.get("MARCA")) or str(row.get("MARCA")).strip() == "":
        motivos.append("MARCA vacía")
    if pd.isna(row.get("MODELO")) or str(row.get("MODELO")).strip() == "":
        motivos.append("MODELO vacío")

    # 3. Validar URL
    url = str(row.get("URL_FICHA")).strip() if pd.notna(row.get("URL_FICHA")) else ""
    if not url.startswith("https"):
        motivos.append(f"URL no válida o no empieza con https ('{url[:30]}...')")

    # 4. Validar Pesos
    p_min = a_numero(row.get("PESO_MIN_KG"))
    p_max = a_numero(row.get("PESO_MAX_KG"))

    if pd.isna(p_min):
        motivos.append(f"PESO_MIN_KG no es un número válido ('{row.get('PESO_MIN_KG')}')")
    if pd.isna(p_max):
        motivos.append(f"PESO_MAX_KG no es un número válido ('{row.get('PESO_MAX_KG')}')")
    if pd.notna(p_min) and pd.notna(p_max) and p_max < p_min:
        motivos.append(f"Rango de peso invertido (Min: {p_min} > Max: {p_max})")

    if motivos:
        reporte_descartes.append({
            "Fila_Excel": idx + 2, # +2 por el encabezado y el índice base 0
            "Equipo_Original": eq_original,
            "Marca": row.get("MARCA"),
            "Modelo": row.get("MODELO"),
            "Motivos_Descarte": " | ".join(motivos)
        })

df_descartes = pd.DataFrame(reporte_descartes)
print(f"\nFilas descartadas analizadas: {len(df_descartes)}")
display(df_descartes)

In [ ]:
from pathlib import Path
import unicodedata
import pandas as pd

def normalizar_nombre(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return texto.lower().replace("_", "").replace(" ", "")

archivos_invima = [
    p for p in C.BASE.iterdir()
    if p.is_file()
    and "registros" in normalizar_nombre(p.name)
    and "sanitarios" in normalizar_nombre(p.name)
    and p.suffix.lower() in (".csv", ".xlsx")
]

print("Archivos encontrados:")
for archivo in archivos_invima:
    print(archivo.name)

## 5. Lectura, filtro y clasificación de importaciones
Un archivo con columnas faltantes **no detiene el proceso**: se omite con advertencia y queda en `archivos_leidos`. Revise abajo qué columnas se detectaron; si alguna es incorrecta, fíjela en `C.COL_MANUAL`.

In [ ]:
import pandas as pd

# Cargar el CSV de INVIMA
df_inv = pd.read_csv(
    C.ARCHIVO_INVIMA_CSV,
    dtype=str,
    encoding="utf-8-sig"
)

# Eliminar espacios sobrantes en los encabezados
df_inv.columns = df_inv.columns.str.strip()

# Crear índice de INVIMA
invima = CL.IndiceInvima(
    df_inv,
    "REGISTRO_SANITARIO",
    "PRODUCTO",
    "MARCA",
    "ESTADO_REGISTRO"
)

print("Índice INVIMA creado correctamente")
print("Registros cargados:", len(df_inv))

In [ ]:
from pathlib import Path
import pandas as pd
import json
import hashlib
import os

# ============================================================
# CONFIGURACIÓN DE CACHÉ
# ============================================================

CARPETA_CACHE = C.CARPETA_SALIDA / "cache_clasificacion"
CARPETA_CACHE.mkdir(parents=True, exist_ok=True)

ARCHIVO_MANIFIESTO = CARPETA_CACHE / "manifiesto_clasificacion.json"

# Cambiar esta versión si modificas CL.clasificar() o sus criterios.
VERSION_CLASIFICACION = "3_subpartida_fecha_dian_20261121"


# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def obtener_firma(ruta):
    """Identifica un archivo mediante su tamaño y fecha de modificación."""
    ruta = Path(ruta)

    if not ruta.exists():
        return None

    stat = ruta.stat()

    return {
        "tamano": stat.st_size,
        "modificado": stat.st_mtime_ns
    }


def obtener_firma_referencias():
    """Detecta cambios en las fichas técnicas y en el CSV de INVIMA."""
    return {
        "fichas": obtener_firma(C.ARCHIVO_FICHAS),
        "invima": obtener_firma(C.ARCHIVO_INVIMA_CSV),
        "version": VERSION_CLASIFICACION
    }


def leer_archivos(carpeta):
    for p in sorted(Path(carpeta).rglob("*")):

        if p.name.startswith(("~$", "manifiesto")):
            continue

        if p.suffix.lower() == ".parquet" and ".tmp" not in p.name:
            yield p, pd.read_parquet(p)

        elif p.suffix.lower() in (".csv", ".txt"):
            yield p, pd.read_csv(
                p,
                dtype=str,
                sep=None,
                engine="python",
                encoding="utf-8-sig"
            )


def guardar_parquet_seguro(df, destino):
    """Guarda primero en temporal para evitar archivos incompletos."""
    temporal = destino.with_suffix(".tmp.parquet")
    df.to_parquet(temporal, index=False)
    os.replace(temporal, destino)


# ============================================================
# CARGAR MANIFIESTO
# ============================================================

if ARCHIVO_MANIFIESTO.exists():
    with open(ARCHIVO_MANIFIESTO, "r", encoding="utf-8") as f:
        manifiesto = json.load(f)
else:
    manifiesto = {
        "referencias": None,
        "archivos": {}
    }


firma_referencias = obtener_firma_referencias()

# Si cambiaron las referencias, hay que reprocesar todo.
referencias_cambiaron = (
    manifiesto.get("referencias") != firma_referencias
)

if referencias_cambiaron:
    print("Las fichas técnicas o INVIMA cambiaron.")
    print("Se reprocesarán los archivos de origen.")


# ============================================================
# PROCESAMIENTO INCREMENTAL
# ============================================================

archivos_actualizados = {}
lectura = []
columnas_detectadas = None

procesados = 0
reutilizados = 0
errores = 0

for ruta, df in leer_archivos(C.CARPETA_PARQUET):

    nombre = str(ruta.relative_to(C.CARPETA_PARQUET))
    firma = obtener_firma(ruta)

    identificador = hashlib.md5(
        nombre.encode("utf-8")
    ).hexdigest()

    registro = manifiesto["archivos"].get(nombre, {})

    archivo_buenos = CARPETA_CACHE / f"{identificador}_conservados.parquet"
    archivo_malos = CARPETA_CACHE / f"{identificador}_descartados.parquet"

    # --------------------------------------------------------
    # VERIFICAR SI YA FUE PROCESADO
    # --------------------------------------------------------

    sin_cambios = (
        not referencias_cambiaron
        and registro.get("firma") == firma
        and archivo_buenos.exists()
        and archivo_malos.exists()
    )

    if sin_cambios:
        reutilizados += 1

        lectura.append({
            "archivo": nombre,
            "filas": registro.get("filas", 0),
            "conservadas": registro.get("conservadas", 0),
            "descartadas": registro.get("descartadas", 0),
            "estado": "REUTILIZADO"
        })

        print(f"↪ {nombre}: sin cambios, se reutiliza el resultado.")
        archivos_actualizados[nombre] = registro
        continue

    # --------------------------------------------------------
    # PROCESAR ARCHIVO NUEVO O MODIFICADO
    # --------------------------------------------------------

    try:
        ok, desc, col = CL.clasificar(df, fichas, invima)

        columnas_detectadas = col

        ok["ARCHIVO_ORIGEN"] = nombre
        desc["ARCHIVO_ORIGEN"] = nombre

        guardar_parquet_seguro(ok, archivo_buenos)
        guardar_parquet_seguro(desc, archivo_malos)

        nuevo_registro = {
            "firma": firma,
            "filas": len(df),
            "conservadas": len(ok),
            "descartadas": len(desc)
        }

        archivos_actualizados[nombre] = nuevo_registro

        lectura.append({
            "archivo": nombre,
            "filas": len(df),
            "conservadas": len(ok),
            "descartadas": len(desc),
            "estado": "PROCESADO"
        })

        procesados += 1

        print(
            f"✓ {nombre}: {len(df):,} filas -> "
            f"{len(ok):,} conservadas, "
            f"{len(desc):,} descartadas"
        )

    except Exception as e:

        errores += 1

        lectura.append({
            "archivo": nombre,
            "filas": len(df),
            "conservadas": 0,
            "descartadas": 0,
            "estado": f"ERROR: {e}"
        })

        print(f"⚠ ERROR {nombre}: {e}")


# ============================================================
# GUARDAR MANIFIESTO
# ============================================================

manifiesto["referencias"] = firma_referencias
manifiesto["archivos"] = archivos_actualizados

temporal = ARCHIVO_MANIFIESTO.with_suffix(".tmp")

with open(temporal, "w", encoding="utf-8") as f:
    json.dump(manifiesto, f, ensure_ascii=False, indent=2)

os.replace(temporal, ARCHIVO_MANIFIESTO)


# ============================================================
# CONSOLIDAR RESULTADOS GUARDADOS
# ============================================================

buenos = []
malos = []

for nombre, registro in archivos_actualizados.items():

    identificador = hashlib.md5(
        nombre.encode("utf-8")
    ).hexdigest()

    archivo_buenos = CARPETA_CACHE / f"{identificador}_conservados.parquet"
    archivo_malos = CARPETA_CACHE / f"{identificador}_descartados.parquet"

    if archivo_buenos.exists():
        buenos.append(pd.read_parquet(archivo_buenos))

    if archivo_malos.exists():
        malos.append(pd.read_parquet(archivo_malos))


lectura = pd.DataFrame(lectura)

assert buenos and sum(len(b) for b in buenos), (
    "Sin registros conservados. Revise los errores de procesamiento."
)

conservados = pd.concat(buenos, ignore_index=True)

descartados = (
    pd.concat(malos, ignore_index=True)
    if malos else pd.DataFrame()
)

print("datos buenos")
print(buenos[0].info())

# ============================================================
# RESUMEN
# ============================================================

print("\n" + "=" * 60)
print("RESUMEN DEL PROCESAMIENTO")
print("=" * 60)

print(f"Archivos procesados:  {procesados}")
print(f"Archivos reutilizados: {reutilizados}")
print(f"Archivos con errores:  {errores}")

print(f"\nConservados: {len(conservados):,}")
print(f"Descartados: {len(descartados):,}")

if columnas_detectadas is not None:
    print("\nColumnas detectadas:")
    print(columnas_detectadas)

if not descartados.empty:
    display(
        descartados["DESCARTE_TIPO"]
        .value_counts()
        .rename("registros")
        .to_frame()
    )

display(
    pd.crosstab(
        conservados["EQUIPO_CLASIFICADO"],
        conservados["NIVEL_DE_CONFIANZA"],
        margins=True
    )
)

display(lectura)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))

sns.countplot(
    data=conservados,
    y="EQUIPO_CLASIFICADO",
    order=conservados["EQUIPO_CLASIFICADO"].value_counts().index,
    palette="viridis"
)

plt.title("Distribución de equipos biomédicos", fontsize=16)
plt.xlabel("Cantidad de registros")
plt.ylabel("Equipo")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import files

# ============================================================
# CONSOLIDAR VARIABLES PARA ANÁLISIS
# ============================================================

columnas = {
    "EQUIPO_CLASIFICADO": "EQUIPO",
    "DESCRIPCION_MERCANCIA": "DESCRIPCION",
    "CODIGO_PAIS_PROCEDENCIA": "PAIS_PROCEDENCIA",
    "SUBPARTIDA_ARANCELARIA": "SUBPARTIDA",
    "DEPARTAMENTO_IMPORTADOR": "DEPARTAMENTO_IMPORTADOR",
    "FECHA_PRESENTACION": "FECHA_PRESENTACION"
}

# Verificar columnas disponibles
faltantes = [
    col for col in columnas if col not in conservados.columns
]

if faltantes:
    raise KeyError(f"Columnas no encontradas: {faltantes}")

# Seleccionar y renombrar
datos_exportacion = conservados[list(columnas.keys())].copy()

datos_exportacion.rename(columns=columnas, inplace=True)

# ============================================================
# EXPORTAR EXCEL
# ============================================================

archivo_excel = C.CARPETA_SALIDA / "dataset_equipos_biomedicos.xlsx"

datos_exportacion.to_excel(
    archivo_excel,
    index=False,
    engine="openpyxl"
)

print(f"Archivo generado: {archivo_excel}")
print(f"Registros exportados: {len(datos_exportacion):,}")
print(f"Columnas: {len(datos_exportacion.columns)}")

display(datos_exportacion.head())

# ============================================================
# DESCARGAR A TU COMPUTADOR
# ============================================================

files.download(str(archivo_excel))

In [ ]:
import pandas as pd
from google.colab import files

# ============================================================
# CONSOLIDAR VARIABLES
# ============================================================

columnas = {
    # IDENTIFICACIÓN DEL EQUIPO
    "EQUIPO_CLASIFICADO": "EQUIPO",
    "DESCRIPCION_MERCANCIA": "DESCRIPCION",
    "SUBPARTIDA_ARANCELARIA": "SUBPARTIDA",

    # PROCEDENCIA Y ORIGEN
    "CODIGO_PAIS_PROCEDENCIA": "CODIGO_PAIS_PROCEDENCIA",
    "PAIS_PROCEDENCIA": "PAIS_PROCEDENCIA",
    "CODIGO_PAIS_ORIGEN": "CODIGO_PAIS_ORIGEN",
    "PAIS_ORIGEN": "PAIS_ORIGEN",

    # IMPORTADOR Y EXPORTADOR
    "NOMBRE_IMPORTADOR": "NOMBRE_IMPORTADOR",
    "NIT_IMPORTADOR": "NIT_IMPORTADOR",
    "NOMBRE_EXPORTADOR": "NOMBRE_EXPORTADOR",

    # INFORMACIÓN REGULATORIA
    "NUM_REGISTRO_LICENCIA": "NUM_REGISTRO_LICENCIA",

    # INFORMACIÓN ECONÓMICA
    "VALOR_FOB_USD": "VALOR_FOB_USD",
    "VALOR_CIF_USD": "VALOR_CIF_USD",
    "VALOR_ADUANA_USD": "VALOR_ADUANA_USD",
    "CANTIDAD": "CANTIDAD",
    "UNIDAD_COMERCIAL": "UNIDAD_COMERCIAL",
    "TASA_CAMBIO": "TASA_CAMBIO",

    # INFORMACIÓN TEMPORAL
    "FECHA_PRESENTACION": "FECHA_PRESENTACION",
    "FECHA_FACTURA": "FECHA_FACTURA",
    "NUMERO_FACTURA": "NUMERO_FACTURA",

    # INFORMACIÓN GEOGRÁFICA
    "DEPARTAMENTO_IMPORTADOR": "DEPARTAMENTO_IMPORTADOR",
    "DEPARTAMENTO_DESTINO": "DEPARTAMENTO_DESTINO"
}

# ============================================================
# VERIFICAR COLUMNAS
# ============================================================

disponibles = [
    col for col in columnas if col in conservados.columns
]

faltantes = [
    col for col in columnas if col not in conservados.columns
]

if faltantes:
    print("Columnas no disponibles:")
    print(faltantes)

# ============================================================
# EXTRAER Y RENOMBRAR
# ============================================================

datos_exportacion = conservados[disponibles].copy()

datos_exportacion.rename(
    columns=columnas,
    inplace=True
)

# ============================================================
# EXPORTAR EXCEL
# ============================================================

archivo_excel = (
    C.CARPETA_SALIDA / "dataset_equipos_biomedicos.xlsx"
)

datos_exportacion.to_excel(
    archivo_excel,
    index=False,
    engine="openpyxl"
)

print(f"\nArchivo generado: {archivo_excel}")
print(f"Registros: {len(datos_exportacion):,}")
print(f"Variables: {len(datos_exportacion.columns)}")

display(datos_exportacion.head())

# ============================================================
# DESCARGAR
# ============================================================

files.download(str(archivo_excel))

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

archivo_real = Path(
    "/content/drive/MyDrive/REGISTROS_IMPORTACIONES/REGISTROS_SANITARIOS_DE_DISPOSITIVOS_MEDICOS_Y_OTRAS_TECNOLOGIAS_20261116.csv"
)

# ============================================================
# CARGAR ARCHIVO
# ============================================================

if archivo_real.exists():

    print("¡Archivo encontrado con éxito!")
    print("Ruta:", archivo_real)

    df_inv = pd.read_csv(
        archivo_real,
        dtype=str,
        encoding="utf-8-sig",
        low_memory=False
    )

    print("\nDimensiones:", df_inv.shape)
    print("\nColumnas:")
    print(df_inv.columns.tolist())

    display(df_inv.head(3))

else:
    print("No se encontró el archivo.")
    print("Verifica que la unidad G: esté montada y que la ruta sea correcta.")

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata

def normalizar(texto):
    texto = str(texto).lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()


# Limpiar nombres de columnas
df_inv.columns = [
    str(c).strip().strip('"').upper()
    for c in df_inv.columns
]


# =========================================================
# PATRONES DE INCLUSIÓN
# =========================================================

patrones_inclusion = {

    "Tomógrafo": [
        r"\btomografo\b",
        r"\btomografia computarizada\b",
        r"\btomografia axial computarizada\b",
        r"\bcomputed tomography\b",
        r"\bct scanner\b",
        r"\bscanner tomografico\b",
        r"\bequipo de tomografia\b",
        r"\btomografo computarizado\b"
    ],

    "Mesa de cirugía": [
        r"\bmesa quirurgica\b",
        r"\bmesa de cirugia\b",
        r"\bmesa de operaciones\b",
        r"\bmesa operatoria\b",
        r"\bmesa quirurgica electrohidraulica\b",
        r"\bmesa quirurgica electromecanica\b",
        r"\boperating table\b",
        r"\bsurgical table\b"
    ],

    "Ventilador mecánico": [
        r"\bventilador mecanico\b",
        r"\bventilador pulmonar\b",
        r"\bventilador de cuidados intensivos\b",
        r"\bventilador para paciente\b",
        r"\bventilacion mecanica\b",
        r"\bmechanical ventilator\b",
        r"\bventilator\b"
    ],

    "Acelerador lineal": [
        r"\bacelerador lineal\b",
        r"\bacelerador de particulas\b",
        r"\bacelerador de electrones\b",
        r"\blinear accelerator\b",
        r"\blinac\b",
        r"\bequipo de radioterapia\b",
        r"\bsistema de radioterapia\b"
    ],

    "Rayos X": [
        r"\bequipo de rayos x\b",
        r"\bequipos de rayos x\b",
        r"\bmaquina de rayos x\b",
        r"\baparato de rayos x\b",
        r"\bequipo radiologico\b",
        r"\bequipos radiologicos\b",
        r"\bunidad radiologica\b",
        r"\bequipo radiografico\b",
        r"\bequipos radiograficos\b",
        r"\baparato radiografico\b",
        r"\bequipo de radiografia convencional\b",
        r"\bequipo de radiografia digital\b",
        r"\bequipo de radiografia fija\b",
        r"\bequipo de radiologia convencional\b",
        r"\bequipo de diagnostico radiologico\b",
        r"\bx ray machine\b",
        r"\bx ray equipment\b",
        r"\bradiographic equipment\b",
        r"\bradiography equipment\b",
        r"\bdiagnostic x ray equipment\b",
        r"\bconventional radiography equipment\b",
        r"\bdigital radiography equipment\b"
    ]
}


# =========================================================
# PATRONES DE EXCLUSIÓN
# =========================================================

patrones_exclusion = {

    "Tomógrafo": [
        r"\bjeringa\b",
        r"\bjeringas\b",
        r"\binjector\b",
        r"\binjectores\b",
        r"\binyeccion de contraste\b",
        r"\bmedio de contraste\b",
        r"\bcontraste radiologico\b",
        r"\bsoftware\b",
        r"\baplicacion\b",
        r"\bprograma\b",
        r"\bconsumible\b",
        r"\baccesorio\b",
        r"\brepuesto\b",
        r"\bcomponente\b",
        r"\bphantom\b"
    ],

    "Mesa de cirugía": [
        r"\bcolchon\b",
        r"\bcolchones\b",
        r"\bcojin\b",
        r"\bcojines\b",
        r"\bposicionador\b",
        r"\bposicionadores\b",
        r"\bsoporte\b",
        r"\bsoportes\b",
        r"\b accesorio\b",
        r"\brepuesto\b",
        r"\bcomponente\b",
        r"\bcontrol remoto\b",
        r"\bpedal\b",
        r"\bsoftware\b"
    ],

    "Ventilador mecánico": [
        r"\bcircuito respiratorio\b",
        r"\bcircuitos respiratorios\b",
        r"\bmanguera\b",
        r"\bmangueras\b",
        r"\bmascara\b",
        r"\bmascaras\b",
        r"\bcanula\b",
        r"\bcanulas\b",
        r"\bvalvula\b",
        r"\bvalvulas\b",
        r"\bfiltro\b",
        r"\bfiltros\b",
        r"\bhumidificador\b",
        r"\bhumidificadores\b",
        r"\bsoftware\b",
        r"\brepuesto\b",
        r"\bcomponente\b",
        r"\baccesorio\b"
    ],

    "Acelerador lineal": [
        r"\bsoftware\b",
        r"\bprograma\b",
        r"\bplanificacion\b",
        r"\bplanning\b",
        r"\bdosimetria\b",
        r"\bdosimetro\b",
        r"\bdosimetros\b",
        r"\bphantom\b",
        r"\bdetector\b",
        r"\bdetectores\b",
        r"\bmodulo\b",
        r"\bmodulos\b",
        r"\bcomponente\b",
        r"\brepuesto\b",
        r"\baccesorio\b",
        r"\bservicio\b"
    ],

    "Rayos X": [
        r"\bpelicula radiografica\b",
        r"\bpeliculas radiograficas\b",
        r"\bfilm radiographic\b",
        r"\bpapel radiografico\b",
        r"\bplaca radiografica\b",
        r"\bplacas radiograficas\b",
        r"\bdetector radiografico\b",
        r"\bdetectores radiograficos\b",
        r"\bdetector digital\b",
        r"\bpanel detector\b",
        r"\bpaneles detectores\b",
        r"\bchasis radiografico\b",
        r"\bchasis radiograficos\b",
        r"\bcasete radiografico\b",
        r"\bcasetes radiograficos\b",
        r"\bintensificador de imagen\b",
        r"\btubo de rayos x\b",
        r"\btubos de rayos x\b",
        r"\bsoftware\b",
        r"\bprograma\b",
        r"\bprocesamiento de imagen\b",
        r"\bprocesador de imagen\b",
        r"\bprocesadores de imagen\b",
        r"\bmonitor\b",
        r"\bmonitores\b",
        r"\bimpresora\b",
        r"\bimpresoras\b",
        r"\bconsumible\b",
        r"\bconsumibles\b",
        r"\baccesorio\b",
        r"\baccesorios\b",
        r"\brepuesto\b",
        r"\brepuestos\b",
        r"\bcomponente\b",
        r"\bcomponentes\b",
        r"\bimplante\b",
        r"\bimplantes\b",
        r"\bstent\b",
        r"\bstents\b",
        r"\bdental\b",
        r"\bodontologico\b",
        r"\bmamografia\b",
        r"\bmamografo\b",
        r"\bportatil\b",
        r"\bportable\b",
        r"\bmovil\b",
        r"\bmobile\b",
        r"\bmobile x ray\b",
        r"\bportable x ray\b",
        r"\barco en c\b",
        r"\bc arm\b"
    ]
}


# =========================================================
# PREPARAR TEXTO
# =========================================================

campos = [
    "PRODUCTO",
    "MARCA",
    "PRESENTACION_COMERCIAL"
]

for col in campos:
    if col not in df_inv.columns:
        df_inv[col] = ""

df_inv["PRODUCTO_NORMALIZADO"] = (
    df_inv["PRODUCTO"]
    .fillna("")
    .apply(normalizar)
)

df_inv["MARCA_NORMALIZADA"] = (
    df_inv["MARCA"]
    .fillna("")
    .apply(normalizar)
)

df_inv["PRESENTACION_NORMALIZADA"] = (
    df_inv["PRESENTACION_COMERCIAL"]
    .fillna("")
    .apply(normalizar)
)


# =========================================================
# CLASIFICACIÓN ESTRICTA
# =========================================================

def clasificar_invima(producto):

    texto = normalizar(producto)

    if not texto or texto == "nan":
        return []

    coincidencias = []

    for equipo, inclusiones in patrones_inclusion.items():

        # Debe encontrar al menos un término de inclusión
        encontrado = any(
            re.search(patron, texto)
            for patron in inclusiones
        )

        if not encontrado:
            continue

        # Si encuentra un término de exclusión, se descarta
        excluido = any(
            re.search(patron, texto)
            for patron in patrones_exclusion.get(equipo, [])
        )

        if not excluido:
            coincidencias.append(equipo)

    return coincidencias


df_inv["EQUIPOS_RELACIONADOS"] = (
    df_inv["PRODUCTO_NORMALIZADO"]
    .apply(clasificar_invima)
)


# =========================================================
# EXPANDIR Y FILTRAR
# =========================================================

invima_equipos = df_inv.explode(
    "EQUIPOS_RELACIONADOS"
)

invima_equipos = invima_equipos[
    invima_equipos["EQUIPOS_RELACIONADOS"].notna()
].copy()


# =========================================================
# RESULTADOS
# =========================================================

print(
    "Registros INVIMA relacionados:",
    len(invima_equipos)
)

print("\nCantidad por categoría:")

display(
    invima_equipos[
        "EQUIPOS_RELACIONADOS"
    ].value_counts()
)

display(
    invima_equipos[
        [
            "EQUIPOS_RELACIONADOS",
            "PRODUCTO",
            "MARCA",
            "PRESENTACION_COMERCIAL",
            "REGISTRO_SANITARIO",
            "ESTADO_REGISTRO"
        ]
    ].head(20)
)

In [ ]:
resumen = (
    invima_equipos
    .groupby(["EQUIPOS_RELACIONADOS", "ESTADO_REGISTRO"])
    ["REGISTRO_SANITARIO"]
    .nunique()
    .reset_index(name="CANTIDAD")
)

fig = px.bar(
    resumen,
    x="EQUIPOS_RELACIONADOS",
    y="CANTIDAD",
    color="ESTADO_REGISTRO",
    barmode="group",
    text="CANTIDAD",
    title="Registros sanitarios INVIMA por equipo biomédico",
    labels={
        "EQUIPOS_RELACIONADOS": "Equipo biomédico",
        "CANTIDAD": "Registros sanitarios",
        "ESTADO_REGISTRO": "Estado del registro"
    }
)

fig.update_layout(
    xaxis_title="Equipo biomédico",
    yaxis_title="Cantidad de registros",
    legend_title="Estado",
    template="plotly_white"
)

fig.show()

In [ ]:
print("Columnas:")
print(df_inv.columns.tolist())

print("\nPrimeros registros:")
display(df_inv.head(5))

print("\nValores únicos por variable:")
for col in ["GRUPO", "PRODUCTO", "MARCA"]:
    print(f"\n{col}:")
    print("No nulos:", df_inv[col].notna().sum())
    print("Valores únicos:", df_inv[col].nunique())
    display(df_inv[col].value_counts().head(10))

In [ ]:

import pandas as pd
import re
import ast
import unicodedata

# 1. Normalización de texto
def normalizar(texto):
    texto = str(texto).upper().strip()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        c for c in texto if not unicodedata.combining(c)
    )
    return texto


# 2. Diccionario de equipos y denominaciones
equipos = {
    "ECOGRAFO": [
        "ecografo", "ecografos", "equipo de ecografia",
        "equipos de ecografia", "sistema de ecografia",
        "ultrasonido diagnostico", "ultrasonografia",
        "ultrasonido medico", "sistema de ultrasonido",
        "equipo de ultrasonido", "diagnostico por ultrasonido",
        "doppler", "doppler color", "ultrasonido doppler",
        "ecografo doppler", "ultrasound system",
        "diagnostic ultrasound", "ultrasound scanner",
        "ultrasonic diagnostic equipment", "sonography system",
        "voluson", "logiq", "acuson", "epiq", "affiniti",
        "rs85", "hs70", "resona", "mindray ultrasound"
    ],

    "ACELERADOR_LINEAL": [
        "acelerador lineal", "aceleradores lineales",
        "acelerador para radioterapia",
        "acelerador de particulas para radioterapia",
        "sistema de radioterapia", "equipo de radioterapia",
        "radioterapia externa", "terapia de radiacion",
        "terapia de haz externo", "radiacion oncologica",
        "radioterapia conformacional",
        "radioterapia de intensidad modulada",
        "imrt", "vmat", "radiocirugia",
        "linear accelerator", "medical linac", "linac",
        "radiotherapy system", "radiation therapy system",
        "external beam radiation therapy",
        "truebeam", "halcyon", "versa hd",
        "elekta infinity", "elekta synergy", "elekta evo",
        "tomotherapy", "radixact"
    ],

    "MESA_CIRUGIA": [
        "mesa de cirugia", "mesa quirurgica",
        "mesas quirurgicas", "mesa de quirofano",
        "mesa para quirofano", "mesa de operaciones",
        "mesa operatoria", "mesa para operaciones",
        "mesa de intervenciones", "mesa electrohidraulica",
        "mesa electromecanica", "mesa hidraulica quirurgica",
        "mesa quirurgica movil",
        "mesa quirurgica radiotransparente",
        "mesa de cirugia universal",
        "surgical table", "operating table", "operation table",
        "electrohydraulic operating table",
        "electromechanical surgical table",
        "radiolucent surgical table",
        "alphamaxx", "maquet magnus", "trusystem",
        "hybase", "steris 5085", "steris 4085"
    ],

    "RAYOS_X": [
        "rayos x", "equipo de rayos x",
        "sistema de rayos x", "equipo de radiografia",
        "sistema de radiografia", "radiografia digital",
        "radiologia digital", "radiografia convencional",
        "equipo radiologico", "sistema radiologico",
        "sala de rayos x", "sala de radiologia",
        "radiografia computarizada", "radiografia directa",
        "detector digital de radiografia",
        "sistema de radiografia digital"
    ]
}


# 3. Patrones de inclusión
patrones = {
    equipo: re.compile(
        "|".join(
            re.escape(normalizar(termino))
            for termino in terminos
        )
    )
    for equipo, terminos in equipos.items()
}


# 4. Términos que pueden indicar accesorios o productos complementarios
patron_exclusion = re.compile(
    r"SOFTWARE|JERINGA|CATETER|SONDA|TUBO|CIRCUITO|"
    r"FILTRO|CANULA|AGUJA|CLIP|IMPLANTE|CONSUMIBLE|"
    r"REPUESTO|ACCESORIO|ADAPTADOR|CABLE|ELECTRODO|"
    r"MANGUERA|FUNDA|KIT|DESECHABLE|APOSITO|GEL|"
    r"CREMA|EPIDURAL|TRAQUEOSTOM|"
    r"TRANSDUCTOR|PROTECTOR|BATERIA|FUENTE DE PODER"
)


# 5. Preparar información
datos = df_inv.copy()

datos["PRODUCTO_NORMALIZADO"] = (
    datos["PRODUCTO"]
    .fillna("")
    .apply(normalizar)
)


# 6. Identificar equipos por producto
def identificar_equipos(producto):
    coincidencias = [
        equipo
        for equipo, patron in patrones.items()
        if patron.search(producto)
    ]
    return coincidencias


datos["EQUIPOS_IDENTIFICADOS"] = (
    datos["PRODUCTO_NORMALIZADO"]
    .apply(identificar_equipos)
)


# 7. Expandir las coincidencias
datos = datos.explode("EQUIPOS_IDENTIFICADOS")

datos = datos[
    datos["EQUIPOS_IDENTIFICADOS"].notna()
].copy()


# 8. Clasificar candidatos y complementarios
datos["ES_COMPLEMENTARIO"] = (
    datos["PRODUCTO_NORMALIZADO"]
    .str.contains(patron_exclusion, na=False)
)


datos["CLASIFICACION"] = "CANDIDATO_REVISION"

datos.loc[
    ~datos["ES_COMPLEMENTARIO"],
    "CLASIFICACION"
] = "CANDIDATO_EQUIPO_PRINCIPAL"

datos.loc[
    datos["ES_COMPLEMENTARIO"],
    "CLASIFICACION"
] = "ACCESORIO_O_COMPLEMENTARIO"


# 9. Separar resultados
equipos_candidatos = datos[
    datos["CLASIFICACION"] == "CANDIDATO_EQUIPO_PRINCIPAL"
].copy()

accesorios = datos[
    datos["CLASIFICACION"] == "ACCESORIO_O_COMPLEMENTARIO"
].copy()


# 10. Resumen
print("DISTRIBUCIÓN DE CANDIDATOS POR EQUIPO")
display(
    datos.groupby(
        ["EQUIPOS_IDENTIFICADOS", "CLASIFICACION"]
    ).size().unstack(fill_value=0)
)


print("\nCANDIDATOS A EQUIPOS PRINCIPALES")
display(
    equipos_candidatos[
        [
            "PRODUCTO",
            "MARCA",
            "REGISTRO_SANITARIO",
            "ESTADO_REGISTRO",
            "EQUIPOS_IDENTIFICADOS"
        ]
    ].drop_duplicates().head(100)
)


print("\nACCESORIOS Y COMPLEMENTARIOS")
display(
    accesorios[
        [
            "PRODUCTO",
            "MARCA",
            "REGISTRO_SANITARIO",
            "EQUIPOS_IDENTIFICADOS"
        ]
    ].drop_duplicates().head(50)
)


In [ ]:

# ============================================================
# 1. LIBRERÍAS
# ============================================================

import pandas as pd
import numpy as np
import re
import ast
import unicodedata
import plotly.express as px

try:
    from rapidfuzz import fuzz
except ImportError:
    %pip install rapidfuzz -q
    from rapidfuzz import fuzz


# ============================================================
# 2. CONFIGURACIÓN DEL PERIODO
# ============================================================

FECHA_INICIO = pd.Timestamp("2026-01-01")
FECHA_FIN = pd.Timestamp("2026-08-31")

UMBRAL_SIMILITUD = 88


# ============================================================
# 3. NORMALIZACIÓN
# ============================================================

def normalizar(texto):
    if pd.isna(texto):
        return ""

    texto = str(texto).upper().strip()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        c for c in texto
        if not unicodedata.combining(c)
    )
    texto = re.sub(r"[^A-Z0-9]+", " ", texto)

    return re.sub(r"\s+", " ", texto).strip()


# ============================================================
# 4. PATRONES DE IDENTIFICACIÓN DE EQUIPOS INVIMA
# ============================================================

equipos = {
    "ECOGRAFO": [
        "ecografo", "ecografos", "equipo de ecografia",
        "equipos de ecografia", "sistema de ecografia",
        "ultrasonido diagnostico", "ultrasonografia",
        "ultrasonido medico", "sistema de ultrasonido",
        "equipo de ultrasonido", "diagnostico por ultrasonido",
        "doppler", "doppler color", "ultrasonido doppler",
        "ecografo doppler", "ultrasound system",
        "diagnostic ultrasound", "ultrasound scanner",
        "ultrasonic diagnostic equipment", "sonography system",
        "voluson", "logiq", "acuson", "epiq", "affiniti",
        "rs85", "hs70", "resona", "mindray ultrasound"
    ],

    "ACELERADOR_LINEAL": [
        "acelerador lineal", "aceleradores lineales",
        "acelerador para radioterapia",
        "acelerador de particulas para radioterapia",
        "sistema de radioterapia", "equipo de radioterapia",
        "radioterapia externa", "terapia de radiacion",
        "terapia de haz externo", "radiacion oncologica",
        "radioterapia conformacional",
        "radioterapia de intensidad modulada",
        "imrt", "vmat", "radiocirugia",
        "linear accelerator", "medical linac", "linac",
        "radiotherapy system", "radiation therapy system",
        "external beam radiation therapy",
        "truebeam", "halcyon", "versa hd",
        "elekta infinity", "elekta synergy", "elekta evo",
        "tomotherapy", "radixact"
    ],

    "MESA_CIRUGIA": [
        "mesa de cirugia", "mesa quirurgica",
        "mesas quirurgicas", "mesa de quirofano",
        "mesa para quirofano", "mesa de operaciones",
        "mesa operatoria", "mesa para operaciones",
        "mesa de intervenciones", "mesa electrohidraulica",
        "mesa electromecanica", "mesa hidraulica quirurgica",
        "mesa quirurgica movil",
        "mesa quirurgica radiotransparente",
        "mesa de cirugia universal",
        "surgical table", "operating table", "operation table",
        "electrohydraulic operating table",
        "electromechanical surgical table",
        "radiolucent surgical table",
        "alphamaxx", "maquet magnus", "trusystem",
        "hybase", "steris 5085", "steris 4085"
    ],

    "RAYOS_X": [
        "rayos x", "equipo de rayos x",
        "sistema de rayos x", "equipo de radiografia",
        "sistema de radiografia", "radiografia digital",
        "radiologia digital", "radiografia convencional",
        "equipo radiologico", "sistema radiologico",
        "sala de rayos x", "sala de radiologia",
        "radiografia computarizada", "radiografia directa",
        "detector digital de radiografia",
        "sistema de radiografia digital"
    ]
}

patrones = {
    equipo: re.compile(
        "|".join(
            re.escape(normalizar(termino))
            for termino in terminos
        )
    )
    for equipo, terminos in equipos.items()
}


# ============================================================
# 5. PREPARAR INVIMA
# ============================================================

invima = df_inv.copy()

invima["PRODUCTO_NORMALIZADO"] = (
    invima["PRODUCTO"].fillna("").apply(normalizar)
)

invima["MARCA_NORMALIZADA"] = (
    invima["MARCA"].fillna("").apply(normalizar)
)

invima["FECHA_VENCIMIENTO"] = pd.to_datetime(
    invima["FECHA_VENCIMIENTO"],
    errors="coerce",
    dayfirst=True
)

invima["ESTADO_NORMALIZADO"] = (
    invima["ESTADO_REGISTRO"]
    .fillna("")
    .apply(normalizar)
)


# Identificar categoría por denominación del producto
def identificar_categoria(producto):
    coincidencias = [
        categoria
        for categoria, patron in patrones.items()
        if patron.search(producto)
    ]

    # Si hay varias categorías, se conserva para revisión
    if len(coincidencias) == 1:
        return coincidencias[0]

    return None


invima["CATEGORIA"] = (
    invima["PRODUCTO_NORMALIZADO"]
    .apply(identificar_categoria)
)


# Conservar solamente registros vigentes en el corte
invima = invima[
    (invima["ESTADO_NORMALIZADO"] == "VIGENTE") &
    (invima["FECHA_VENCIMIENTO"].notna()) &
    (invima["CATEGORIA"].notna())
].copy()


# Texto que se utilizará para comparar con DIAN
invima["TEXTO_COMPARACION"] = (
    invima["MARCA_NORMALIZADA"] + " " +
    invima["PRODUCTO_NORMALIZADO"]
).str.strip()


# Evitar duplicados de registros sanitarios
invima = invima.drop_duplicates(
    subset=["REGISTRO_SANITARIO", "CATEGORIA", "TEXTO_COMPARACION"]
).reset_index(drop=True)


# ============================================================
# 6. PREPARAR DIAN
# ============================================================

dian = conservados.copy()

dian["FECHA_PRESENTACION"] = pd.to_datetime(
    dian["FECHA_PRESENTACION"],
    errors="coerce",
    dayfirst=True
)

dian = dian[
    dian["FECHA_PRESENTACION"].between(
        FECHA_INICIO,
        FECHA_FIN
    )
].copy()

dian["DESCRIPCION_NORMALIZADA"] = (
    dian["DESCRIPCION_MERCANCIA"]
    .fillna("")
    .apply(normalizar)
)

dian["CATEGORIA"] = (
    dian["EQUIPO_CLASIFICADO"]
    .fillna("")
    .apply(normalizar)
)

# Homologación de categorías
equivalencias = {
    "ECOGRAFO": "ECOGRAFO",
    "ECOGRAFOS": "ECOGRAFO",
    "ACELERADOR LINEAL": "ACELERADOR_LINEAL",
    "ACELERADORES LINEALES": "ACELERADOR_LINEAL",
    "ACELERADOR_LINEAL": "ACELERADOR_LINEAL",
    "MESA CIRUGIA": "MESA_CIRUGIA",
    "MESA DE CIRUGIA": "MESA_CIRUGIA",
    "MESA_CIRUGIA": "MESA_CIRUGIA",
    "RAYOS X": "RAYOS_X",
    "RAYOS_X": "RAYOS_X"
}

dian["CATEGORIA"] = dian["CATEGORIA"].replace(equivalencias)

dian = dian[
    dian["CATEGORIA"].isin(equipos.keys())
].copy()


# ============================================================
# 7. CRUCE DIAN - INVIMA
# ============================================================

coincidencias = []

for categoria in equipos.keys():

    inv_categoria = invima[
        invima["CATEGORIA"] == categoria
    ].copy()

    dian_categoria = dian[
        dian["CATEGORIA"] == categoria
    ].copy()

    if inv_categoria.empty or dian_categoria.empty:
        continue

    registros_invima = inv_categoria.to_dict("records")

    for _, fila_dian in dian_categoria.iterrows():

        descripcion_dian = fila_dian["DESCRIPCION_NORMALIZADA"]

        if not descripcion_dian:
            continue

        mejor_coincidencia = None
        mejor_puntaje = 0

        for fila_invima in registros_invima:

            # La fecha de vencimiento debe ser posterior
            # o igual a la fecha de presentación DIAN
            if (
                fila_invima["FECHA_VENCIMIENTO"]
                < fila_dian["FECHA_PRESENTACION"]
            ):
                continue

            texto_invima = fila_invima["TEXTO_COMPARACION"]

            puntaje = fuzz.token_set_ratio(
                descripcion_dian,
                texto_invima
            )

            if puntaje > mejor_puntaje:
                mejor_puntaje = puntaje
                mejor_coincidencia = fila_invima

        if (
            mejor_coincidencia is not None
            and mejor_puntaje >= UMBRAL_SIMILITUD
        ):

            coincidencias.append({
                "CATEGORIA": categoria,
                "FECHA_PRESENTACION": fila_dian["FECHA_PRESENTACION"],
                "DESCRIPCION_DIAN": fila_dian["DESCRIPCION_MERCANCIA"],
                "MARCA_INVIMA": mejor_coincidencia["MARCA"],
                "PRODUCTO_INVIMA": mejor_coincidencia["PRODUCTO"],
                "REGISTRO_SANITARIO": mejor_coincidencia["REGISTRO_SANITARIO"],
                "ESTADO_REGISTRO": mejor_coincidencia["ESTADO_REGISTRO"],
                "FECHA_VENCIMIENTO": mejor_coincidencia["FECHA_VENCIMIENTO"],
                "NOMBRE_IMPORTADOR": fila_dian.get("NOMBRE_IMPORTADOR"),
                "NIT_IMPORTADOR": fila_dian.get("NIT_IMPORTADOR"),
                "VALOR_CIF_USD": fila_dian.get("VALOR_CIF_USD"),
                "CANTIDAD": fila_dian.get("CANTIDAD"),
                "SIMILITUD": mejor_puntaje
            })


# ============================================================
# 8. DATAFRAME FINAL DE COINCIDENCIAS
# ============================================================

df_coincidencias = pd.DataFrame(coincidencias)

if df_coincidencias.empty:

    print("No se encontraron coincidencias con el umbral establecido.")

else:

    df_coincidencias = df_coincidencias.drop_duplicates(
        subset=[
            "CATEGORIA",
            "FECHA_PRESENTACION",
            "DESCRIPCION_DIAN",
            "REGISTRO_SANITARIO"
        ]
    )

    print("TOTAL DE COINCIDENCIAS:", len(df_coincidencias))

    print("\nCOINCIDENCIAS POR CATEGORÍA:")
    display(
        df_coincidencias["CATEGORIA"]
        .value_counts()
        .rename_axis("EQUIPO")
        .reset_index(name="COINCIDENCIAS")
    )

    display(df_coincidencias.head(30))


    # ========================================================
    # 9. GRÁFICO DE MARCAS Y MODELOS COINCIDENTES
    # ========================================================

    df_grafico = df_coincidencias.copy()

    df_grafico["MARCA_INVIMA"] = (
        df_grafico["MARCA_INVIMA"]
        .fillna("Marca no especificada")
        .replace("", "Marca no especificada")
    )

    df_grafico["MODELO"] = (
        df_grafico["PRODUCTO_INVIMA"]
        .fillna("Modelo no especificado")
    )

    df_grafico["MARCA_MODELO"] = (
        df_grafico["MARCA_INVIMA"].astype(str)
        + " | "
        + df_grafico["MODELO"].astype(str)
    )

    resumen = (
        df_grafico
        .groupby(
            ["CATEGORIA", "MARCA_MODELO"],
            as_index=False
        )
        .agg(
            COINCIDENCIAS=("REGISTRO_SANITARIO", "count"),
            REGISTROS_INVIMA=("REGISTRO_SANITARIO", "nunique")
        )
        .sort_values("COINCIDENCIAS", ascending=False)
    )

    # Top 15 por categoría
    top_modelos = (
        resumen
        .sort_values("COINCIDENCIAS", ascending=False)
        .groupby("CATEGORIA")
        .head(15)
    )

    fig = px.bar(
        top_modelos,
        x="COINCIDENCIAS",
        y="MARCA_MODELO",
        color="CATEGORIA",
        facet_col="CATEGORIA",
        facet_col_wrap=2,
        orientation="h",
        text="COINCIDENCIAS",
        hover_data=["REGISTROS_INVIMA"],
        title=(
            "Marcas y modelos con coincidencia "
            "entre INVIMA y DIAN (enero-agosto 2026)"
        ),
        labels={
            "COINCIDENCIAS": "Coincidencias DIAN-INVIMA",
            "MARCA_MODELO": "Marca | Modelo",
            "CATEGORIA": "Categoría",
            "REGISTROS_INVIMA": "Registros sanitarios distintos"
        },
        height=1100
    )

    fig.update_layout(
        template="plotly_white",
        showlegend=False,
        title_x=0.5,
        font=dict(size=11)
    )

    fig.update_traces(
        textposition="outside",
        cliponaxis=False
    )

    fig.update_yaxes(
        autorange="reversed",
        automargin=True
    )

    fig.show()


    # ========================================================
    # 10. EXPORTAR RESULTADOS
    # ========================================================

    df_coincidencias.to_csv(
        "coincidencias_INVIMA_DIAN_2026.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("\nArchivo exportado: coincidencias_INVIMA_DIAN_2026.csv")


In [ ]:

import pandas as pd
import re

# Conversor de fechas que reconoce YYYYMMDD, DD/MM/YYYY e ISO
def convertir_fecha(valor):
    if pd.isna(valor):
        return pd.NaT

    texto = str(valor).strip()

    if not texto:
        return pd.NaT

    # Formato DIAN: YYYYMMDD
    if re.fullmatch(r"\d{8}", texto):
        return pd.to_datetime(
            texto,
            format="%Y%m%d",
            errors="coerce"
        )

    # Formato YYYY-MM-DD
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", texto):
        return pd.to_datetime(
            texto,
            format="%Y-%m-%d",
            errors="coerce"
        )

    # Formato DD/MM/YYYY
    if re.fullmatch(r"\d{2}/\d{2}/\d{4}", texto):
        return pd.to_datetime(
            texto,
            format="%d/%m/%Y",
            errors="coerce"
        )

    return pd.to_datetime(texto, errors="coerce")


# =========================================
# DIAGNÓSTICO DIAN
# =========================================

dian_diag = conservados.copy()

dian_diag["FECHA_CONVERTIDA"] = (
    dian_diag["FECHA_PRESENTACION"].apply(convertir_fecha)
)

print("DIAN - FECHAS ORIGINALES")
display(dian_diag["FECHA_PRESENTACION"].head(10))

print("\nDIAN - FECHAS CONVERTIDAS")
display(dian_diag["FECHA_CONVERTIDA"].head(10))

print("\nDIAN - RESUMEN DE FECHAS")
print("Fechas válidas:", dian_diag["FECHA_CONVERTIDA"].notna().sum())
print("Fechas inválidas:", dian_diag["FECHA_CONVERTIDA"].isna().sum())
print("Fecha mínima:", dian_diag["FECHA_CONVERTIDA"].min())
print("Fecha máxima:", dian_diag["FECHA_CONVERTIDA"].max())

dian_2026 = dian_diag[
    (dian_diag["FECHA_CONVERTIDA"] >= "2026-01-01") &
    (dian_diag["FECHA_CONVERTIDA"] < "2026-09-01")
].copy()

print("\nDIAN - REGISTROS ENTRE ENERO Y AGOSTO DE 2026:")
print(len(dian_2026))

print("\nDIAN - CATEGORÍAS:")
display(dian_2026["EQUIPO_CLASIFICADO"].value_counts(dropna=False))


# =========================================
# DIAGNÓSTICO INVIMA
# =========================================

invima_diag = df_inv.copy()

invima_diag["VENCIMIENTO_CONVERTIDO"] = (
    invima_diag["FECHA_VENCIMIENTO"].apply(convertir_fecha)
)

print("\nINVIMA - FECHAS ORIGINALES")
display(invima_diag["FECHA_VENCIMIENTO"].head(10))

print("\nINVIMA - FECHAS CONVERTIDAS")
display(invima_diag["VENCIMIENTO_CONVERTIDO"].head(10))

print("\nINVIMA - RESUMEN DE FECHAS")
print("Fechas válidas:", invima_diag["VENCIMIENTO_CONVERTIDO"].notna().sum())
print("Fechas inválidas:", invima_diag["VENCIMIENTO_CONVERTIDO"].isna().sum())
print("Fecha mínima:", invima_diag["VENCIMIENTO_CONVERTIDO"].min())
print("Fecha máxima:", invima_diag["VENCIMIENTO_CONVERTIDO"].max())

print("\nINVIMA - ESTADOS:")
display(invima_diag["ESTADO_REGISTRO"].value_counts(dropna=False))

print("\nINVIMA - EQUIPOS RELACIONADOS:")
display(invima_diag["EQUIPOS_RELACIONADOS"].value_counts().head(20))


In [ ]:
# Calidad: modelos mencionados con INVIMA/marca pero SIN ficha (qué fichas cargar primero)
c = conservados[conservados['FUENTE_MODELO'] == '']
print('Registros sin ficha técnica coincidente:', len(c), 'de', len(conservados))
display(conservados['VERIFICACION_INVIMA'].value_counts().to_frame())

## 6. Indicadores de precio y machine learning (objetivo 2)
`PRECIO_UNIT_USD = FOB / cantidad` solo si la unidad comercial es individual. ML: anomalías de precio (IsolationForest/MAD), precio esperado con validación cruzada y clasificador de texto con etiquetas débiles. **Las anomalías se marcan, no se borran.**

In [ ]:
base = I.preparar(conservados, solo_principal=True)
base = ml.marcar_anomalias(base)
base, met_precio = ml.precio_esperado(base)
_, met_cat = ml.clasificador_categoria(pd.concat([conservados, descartados], ignore_index=True))
print('Precio esperado:', met_precio); print('Clasificador de categoría:', met_cat)
base['FAMILIA'] = base['EQUIPO_CLASIFICADO'].map({k: v['nombre'] for k, v in C.EQUIPOS.items()})
sin_anom = base[~base['ES_ANOMALIA_PRECIO']]
serie = I.serie_precios(sin_anom, 'Y', por=('FAMILIA',))
display(serie)
px.line(serie, x='PERIODO', y='mediana', color='FAMILIA', markers=True, log_y=True, hover_data=['n', 'p25', 'p75', 'VAR_PCT_MEDIANA'],
        title='Precio unitario mediano (USD, sin anomalías)').show()
px.bar(serie, x='PERIODO', y='VAR_PCT_MEDIANA', color='FAMILIA', barmode='group', title='Variación % anual de la mediana').show()
pais = I.comparacion_pais(sin_anom.assign(EQUIPO_CLASIFICADO=sin_anom['FAMILIA']))
display(pais)
for fam, g in pais.groupby('EQUIPO_CLASIFICADO'):
    px.bar(g, x='PAIS_ORIGEN', y='mediana', color='PRIMA_PCT_VS_FAMILIA', color_continuous_scale='RdBu_r', color_continuous_midpoint=0,
           hover_data=['n', 'CUOTA_UNIDADES_PCT'], title=f'{fam}: precio mediano por país (color = prima % vs familia)').show()

## 7. Actores importadores (objetivo 5)

In [ ]:
dd = sin_anom.assign(EQUIPO_CLASIFICADO=sin_anom['FAMILIA'])
perfil = ml.segmentar_importadores(I.perfil_importadores(dd))
conc = I.concentracion(dd)
display(perfil.groupby('EQUIPO_CLASIFICADO').head(10))
px.line(conc, x='ANIO', y='HHI', color='EQUIPO_CLASIFICADO', markers=True, title='Concentración del mercado (HHI, 0–10.000)').show()
for fam in dd['EQUIPO_CLASIFICADO'].unique():
    px.area(I.evolucion_cuotas(dd, fam), x='ANIO', y='CUOTA_PCT', color='NOMBRE', title=f'{fam}: evolución de cuota de valor FOB').show()

## 8. Licitaciones y contratos públicos (SECOP II)
Fuente: datos.gov.co (`p6dx-8zbt` procesos, `jbjy-vk9h` contratos). **Los nombres de columna se descubren desde los metadatos del dataset**; la celda siguiente imprime cuáles se encontraron para que los verifique.

Columnas estándar resultantes: `ID, ENTIDAD, NIT_ENTIDAD, DEPARTAMENTO, CIUDAD, MODALIDAD, DESCRIPCION, VALOR_ESTIMADO, VALOR_ADJUDICADO, PROVEEDOR, NIT_PROVEEDOR, FECHA, ESTADO, URL` + `FAMILIA, CATEGORIA_LICITACION, ANIO, DATASET, ADVERTENCIA`.

> El valor es del **proceso o contrato** (puede incluir varias unidades, servicios o mantenimiento): no es precio unitario y no hay un identificador común con las declaraciones de importación.

In [ ]:
# 8.1 Verificación de columnas (sin descargar datos)
for ds in (C.SECOP_PROCESOS, C.SECOP_CONTRATOS):
    cols = secop.columnas_dataset(ds)
    m = secop._mapa(cols)
    print(f'\n== {ds}: {len(cols)} columnas')
    print('  mapeadas  :', {k: v for k, v in m.items() if v})
    print('  SIN hallar:', [k for k, v in m.items() if not v])
    print('  todas     :', cols)

In [ ]:
import requests
import time

URL = "https://www.datos.gov.co"

for ds in (C.SECOP_PROCESOS, C.SECOP_CONTRATOS):
    print(f"\nProbando {ds}")

    try:
        t0 = time.time()

        r = requests.get(
            f"{URL}/resource/{ds}.json",
            params={"$limit": 1},
            timeout=30
        )

        print("Estado:", r.status_code)
        print("Tiempo:", round(time.time() - t0, 2), "segundos")
        print("Respuesta:", r.text[:500])

    except Exception as e:
        print("Error:", repr(e))

In [ ]:
import requests
import pandas as pd
import time

datasets = {
    "procesos": C.SECOP_PROCESOS,
    "contratos": C.SECOP_CONTRATOS
}

for nombre, ds in datasets.items():

    print(f"\n--- {nombre.upper()} ---")

    try:
        t0 = time.time()

        r = requests.get(
            f"https://www.datos.gov.co/resource/{ds}.json",
            params={
                "$q": "ecografo",
                "$limit": 20
            },
            timeout=60
        )

        r.raise_for_status()

        datos = r.json()

        print("Tiempo:", round(time.time() - t0, 2), "s")
        print("Registros:", len(datos))

        df_temp = pd.DataFrame(datos)

        if len(df_temp):
            display(df_temp.head(3))

    except Exception as e:
        print("Error:", e)

In [ ]:
import requests
import pandas as pd
import time

ds = C.SECOP_CONTRATOS

t0 = time.time()

r = requests.get(
    f"https://www.datos.gov.co/resource/{ds}.json",
    params={
        "$q": "ecografo",
        "$limit": 20
    },
    timeout=60
)

r.raise_for_status()

datos = r.json()

print("Tiempo:", round(time.time() - t0, 2), "segundos")
print("Registros:", len(datos))

df_contratos_test = pd.DataFrame(datos)

display(df_contratos_test.head())

In [ ]:
LIMITE = 1000

partes = []

for ds in (C.SECOP_PROCESOS, C.SECOP_CONTRATOS):
    print(f"Descargando dataset: {ds}")

    try:
        df_secop = secop.descargar(ds, limite=LIMITE)
        partes.append(df_secop)
        print(f"Descargados: {len(df_secop):,} registros")

    except Exception as e:
        print(f"Error descargando {ds}: {e}")

lic = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()

if len(lic):
    lic['FAMILIA_NOMBRE'] = lic['FAMILIA'].map(
        {k: v['nombre'] for k, v in C.EQUIPOS.items()}
    )

    print(len(lic), 'registros relacionados')
    display(
        pd.crosstab(
            lic['FAMILIA_NOMBRE'],
            [lic['DATASET'], lic['CATEGORIA_LICITACION']]
        )
    )
else:
    print("No se pudieron descargar registros de SECOP II.")

In [ ]:
# 8.3 Análisis: solo equipos principales; contratos (valor firmado) vs procesos (precio base)
eq = lic[lic['CATEGORIA_LICITACION'] == 'Equipo principal completo']
contr = eq[eq['DATASET'] == C.SECOP_CONTRATOS]
por_anio = contr.groupby(['ANIO', 'FAMILIA_NOMBRE'])['VALOR_ESTIMADO'].agg(['count', 'sum', 'median']).reset_index()
display(por_anio)
px.bar(por_anio, x='ANIO', y='sum', color='FAMILIA_NOMBRE', title='Valor de contratos de equipos principales (COP)').show()
prov = contr.groupby(['FAMILIA_NOMBRE', 'PROVEEDOR'])['VALOR_ESTIMADO'].agg(['count', 'sum']).reset_index().sort_values('sum', ascending=False)
display(prov.groupby('FAMILIA_NOMBRE').head(10))
ent = contr.groupby(['FAMILIA_NOMBRE', 'ENTIDAD'])['VALOR_ESTIMADO'].agg(['count', 'sum']).reset_index().sort_values('sum', ascending=False)
display(ent.groupby('FAMILIA_NOMBRE').head(10))

# Comparación indicativa con importaciones (misma familia y año): número de eventos y valor, en monedas distintas (COP vs USD)
imp_anio = sin_anom.groupby(['FAMILIA', 'ANIO']).agg(importaciones=('_fob', 'size'), fob_usd=('_fob', 'sum')).reset_index()
cmp_ = por_anio.rename(columns={'FAMILIA_NOMBRE': 'FAMILIA', 'count': 'contratos', 'sum': 'valor_contratos_cop'}).merge(imp_anio, on=['FAMILIA', 'ANIO'], how='outer')
display(cmp_.sort_values(['FAMILIA', 'ANIO']))

## 9. Pruebas de validación (objetivo 4)

In [ ]:
%%writefile /content/bioprice_bi/tests/test_core.py
"""Casos de prueba (objetivo 4). Ejecutar:  python -m unittest discover -s tests -v   (o pytest)"""
import sys
import unittest
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from bioprice import indicadores as I          # noqa: E402
from bioprice import ml, secop                  # noqa: E402
from bioprice.clasificacion import (a_numero, asignar_equipo, buscar_ficha, claves_registro, clasificar,   # noqa: E402
                                    clasificar_categoria, coincide_marca_modelo, norm, peso_unitario, validar_fichas)


def fichas_demo():
    return validar_fichas(pd.DataFrame([
        {"EQUIPO": "VENTILADOR", "MARCA": "Hamilton Medical", "MODELO": "T1", "PESO_MIN_KG": "6", "PESO_MAX_KG": "7",
         "URL_FICHA": "https://ejemplo.org/t1", "CONDICION_PESO": "con batería", "FECHA_CONSULTA": "2026-10-01"},
        {"EQUIPO": "ECOGRAFO", "MARCA": "GE Healthcare", "MODELO": "LOGIQ E10", "PESO_MIN_KG": "60", "PESO_MAX_KG": "80",
         "URL_FICHA": "https://ejemplo.org/e10", "CONDICION_PESO": "n/d", "FECHA_CONSULTA": "2026-10-01"},
        {"EQUIPO": "RAYOS_X", "MARCA": "X", "MODELO": "Y1", "PESO_MIN_KG": "", "PESO_MAX_KG": "", "URL_FICHA": "", "CONDICION_PESO": "", "FECHA_CONSULTA": ""},
    ]))


def crudo_demo():
    f = [
        ("901920", "VENTILADOR MECANICO HAMILTON MEDICAL T1", "2", "UNIDAD", "13", "16", "30000", "ALEMANIA", "HOSPITAL A", "2024-03-10"),
        ("901920", "ELECTRODOS PARA VENTILADOR MECANICO", "100", "UNIDAD", "5", "6", "300", "CHINA", "HOSPITAL A", "2024-03-10"),
        ("901812", "ECOGRAFO GE HEALTHCARE LOGIQ E10 CON TRANSDUCTOR", "1", "UNIDAD", "70", "90", "60000", "ESTADOS UNIDOS", "CLINICA B", "2023-05-01"),
        ("902221", "ACELERADOR LINEAL VERSA", "1", "UNIDAD", "7000", "9000", "2500000", "SUECIA", "ONCOLOGOS S.A.", "2022-08-01"),
        ("901812", "ECOGRAFO ", "5", "CAJA", "20", "25", "9000", "CHINA", "CLINICA B", "2023-06-01"),
        ("999999", "COMPUTADOR", "1", "UNIDAD", "2", "3", "500", "CHINA", "OFICINA", "2023-06-01"),
    ]
    return pd.DataFrame(f, columns=["SUBPARTIDA", "DESCRIPCION_MERCANCIA", "CANTIDAD", "UNIDAD_COMERCIAL", "PESO_NETO", "PESO_BRUTO",
                                    "VALOR_FOB_USD", "PAIS_ORIGEN", "IMPORTADOR", "FECHA"])


class TestTexto(unittest.TestCase):
    def test_norm_y_numeros(self):
        self.assertEqual(norm("Ecógrafo"), "ECOGRAFO")
        self.assertEqual(a_numero("1.234,5"), 1234.5)
        self.assertEqual(a_numero("1,234.5"), 1234.5)
        self.assertTrue(np.isnan(a_numero("abc")))

    def test_peso_unitario(self):
        self.assertEqual(peso_unitario("UNIDAD", 2, 14)[0], 7)
        self.assertTrue(np.isnan(peso_unitario("CAJA", 2, 14)[0]))
        self.assertTrue(np.isnan(peso_unitario("UNIDAD", 0, 14)[0]))

    def test_registro_citado(self):
        self.assertEqual(claves_registro("INVIMA 2019DM-0001234"), {"2019DM-1234"})


class TestClasificacion(unittest.TestCase):
    def test_equipo_y_categoria(self):
        eq, origen, _ = asignar_equipo("ACELERADOR LINEAL VERSA", "902221")
        self.assertEqual((eq, origen), ("ACELERADOR_LINEAL", "descripcion"))
        self.assertEqual(clasificar_categoria("ELECTRODOS PARA VENTILADOR MECANICO", "VENTILADOR", "descripcion")[0], "Consumible")
        # lo que viene después de «CON» no vuelve accesorio al equipo
        self.assertEqual(clasificar_categoria("ECOGRAFO CON TRANSDUCTOR", "ECOGRAFO", "descripcion")[0], "Equipo principal completo")

    def test_marca_modelo_juntos(self):
        self.assertTrue(coincide_marca_modelo("VENTILADOR HAMILTON-T1 NUEVO", norm("Hamilton"), norm("T1")))
        self.assertFalse(coincide_marca_modelo("VENTILADOR T1 " + "X" * 80 + " HAMILTON", norm("Hamilton"), norm("T1")))
        self.assertFalse(coincide_marca_modelo("VENTILADOR MARCA ACME T1", norm("Hamilton"), norm("T1")))

    def test_fichas_invalidas_descartadas(self):
        self.assertEqual(len(fichas_demo()), 2)

    def test_flujo_completo(self):
        ok, desc, col = clasificar(crudo_demo(), fichas_demo())
        self.assertEqual(col["fob"], "VALOR_FOB_USD")
        self.assertEqual(len(ok) + len(desc), 5)                       # el computador (sub. 999999) no es candidato
        motivos = dict(zip(desc["_descripcion"], desc["DESCARTE_TIPO"]))
        self.assertEqual(motivos["ELECTRODOS PARA VENTILADOR MECANICO"], "Categoría no principal")
        self.assertEqual(motivos["ECOGRAFO PORTATIL"], "Sin peso unitario válido")   # CAJA no es unidad individual
        vent = ok[ok["_descripcion"].str.startswith("VENTILADOR")].iloc[0]
        self.assertAlmostEqual(vent["PESO_UNITARIO_KG"], 6.5)
        self.assertEqual(vent["COINCIDENCIA_PESO"], "Sí")
        self.assertEqual(vent["NIVEL_DE_CONFIANZA"], "Medio")           # sin INVIMA nunca llega a Alto
        self.assertTrue((ok["ESTADO_IDENTIFICACION"] == "Candidato").all())

    def test_ficha_no_se_asigna_por_peso(self):
        self.assertIsNone(buscar_ficha("VENTILADOR MECANICO SIN MARCA", "VENTILADOR", fichas_demo()))


def base_sintetica(n=300, seed=0):
    rng = np.random.default_rng(seed)
    fam = rng.choice(["ECOGRAFO", "VENTILADOR"], n)
    base = np.where(fam == "ECOGRAFO", 30000, 15000)
    anio = rng.integers(2020, 2025, n)
    precio = base * (1 + 0.05 * (anio - 2020)) * rng.lognormal(0, 0.1, n)
    precio[:6] *= 40                                                    # anomalías evidentes
    imp = rng.choice(["A", "B", "C", "D", "E", "F"], n, p=[.4, .25, .15, .1, .05, .05])
    return pd.DataFrame({
        "EQUIPO_CLASIFICADO": fam, "FECHA": pd.to_datetime(anio.astype(str) + "-06-01"), "ANIO": anio, "_cantidad": 1.0, "_fob": precio,
        "_unidad": "UNIDAD", "_pais": rng.choice(["CHINA", "ALEMANIA", "EEUU"], n), "_importador": imp, "_nit": "", "PRECIO_UNIT_USD": precio,
        "PESO_UNITARIO_KG": rng.normal(10, 1, n).clip(1), "MODELO_CANDIDATO": "Pendiente de identificación",
        "CATEGORIA_REGISTRO": "Equipo principal completo", "NIVEL_DE_CONFIANZA": "Medio"})


class TestIndicadores(unittest.TestCase):
    def setUp(self):
        self.d = base_sintetica()

    def test_variacion_y_pais(self):
        s = I.serie_precios(self.d, "Y")
        self.assertTrue(s["VAR_PCT_MEDIANA"].dropna().gt(-50).all())
        self.assertEqual(s.groupby("EQUIPO_CLASIFICADO")["VAR_PCT_MEDIANA"].apply(lambda x: x.isna().sum()).max(), 1)  # solo el 1.er periodo
        p = I.comparacion_pais(self.d)
        self.assertAlmostEqual(p.groupby("EQUIPO_CLASIFICADO")["CUOTA_UNIDADES_PCT"].sum().iloc[0], 100, places=6)

    def test_precio_requiere_unidad_individual(self):
        d = self.d.copy()
        d.loc[d.index[:3], "_unidad"] = "CAJA"
        d = d.drop(columns="PRECIO_UNIT_USD")
        self.assertTrue(I.preparar(d)["PRECIO_UNIT_USD"].iloc[:3].isna().all())

    def test_concentracion_y_actores(self):
        c = I.concentracion(self.d)
        self.assertTrue(((c["HHI"] > 0) & (c["HHI"] <= 10000)).all())
        perfil = I.perfil_importadores(self.d)
        self.assertAlmostEqual(perfil.groupby("EQUIPO_CLASIFICADO")["CUOTA_VALOR_PCT"].sum().iloc[0], 100, places=6)


class TestML(unittest.TestCase):
    def test_detecta_anomalias(self):
        d = ml.marcar_anomalias(base_sintetica(), contaminacion=0.03)
        self.assertGreaterEqual(int(d.iloc[:6]["ES_ANOMALIA_PRECIO"].sum()), 4)   # detecta la mayoría de las 6 sembradas

    def test_precio_esperado_mejora_a_la_media(self):
        d = ml.marcar_anomalias(base_sintetica())
        _, met = ml.precio_esperado(d)
        self.assertEqual(met["estado"], "ok")
        self.assertLess(met["MAE_log_cv"], met["MAE_log_base_media"])

    def test_datos_insuficientes_no_fallan(self):
        _, met = ml.precio_esperado(ml.marcar_anomalias(base_sintetica(20)))
        self.assertTrue(met["estado"].startswith("Insuficiente"))


class TestSecop(unittest.TestCase):
    def test_normalizar_sin_red(self):
        raw = pd.DataFrame({"entidad": ["HOSPITAL X", "ALCALDIA"], "descripcion_del_proceso": ["COMPRA DE VENTILADOR MECANICO", "PAPELERIA"],
                            "valor_del_contrato": ["1000", "5"], "fecha_de_firma": ["2024-01-05", "2024-01-06"], "urlproceso": [{"url": "http://x"}, {"url": "http://y"}]})
        out = secop.normalizar(raw, "jbjy-vk9h")
        self.assertEqual(len(out), 1)
        self.assertEqual(out.loc[0, "FAMILIA"], "VENTILADOR")
        self.assertEqual(out.loc[0, "URL"], "http://x")


if __name__ == "__main__":
    unittest.main()

In [ ]:
!cd /content/bioprice_bi && python -m unittest discover -s tests -v 2>&1 | tail -30

## 10. Exportar resultados

In [ ]:
C.CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
base.to_parquet(C.BASE_ANALITICA, index=False)
descartados.to_csv(C.DESCARTADOS, index=False, encoding='utf-8-sig')
lic.to_csv(C.CARPETA_SALIDA / 'licitaciones_secop.csv', index=False, encoding='utf-8-sig')
import json
json.dump({'precio_esperado': met_precio, 'clasificador_categoria': met_cat}, open(C.CARPETA_SALIDA / 'metricas_ml.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
import re as _re
ILEG = _re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f]')
def limpio(df):
    df = df.copy()
    for c in df.select_dtypes(include='object').columns:
        df[c] = df[c].map(lambda v: ILEG.sub('', v) if isinstance(v, str) else v)
    return df
with pd.ExcelWriter(C.CARPETA_SALIDA / 'resultado_bioprice.xlsx', engine='openpyxl') as w:
    pd.crosstab(base['FAMILIA'], base['NIVEL_DE_CONFIANZA'], margins=True).to_excel(w, sheet_name='Resumen')
    limpio(base[base['ESTADO_IDENTIFICACION'] == 'Confirmado']).to_excel(w, sheet_name='Confirmados', index=False)
    limpio(base[base['ESTADO_IDENTIFICACION'] == 'Candidato']).to_excel(w, sheet_name='Candidatos', index=False)
    serie.to_excel(w, sheet_name='Precio_por_anio', index=False)
    pais.to_excel(w, sheet_name='Por_pais', index=False)
    perfil.to_excel(w, sheet_name='Importadores', index=False)
    conc.to_excel(w, sheet_name='Concentracion', index=False)
    limpio(lic.head(100000)).to_excel(w, sheet_name='Licitaciones', index=False)
    descartados['DESCARTE_TIPO'].value_counts().rename('registros').to_frame().to_excel(w, sheet_name='Resumen_descartes')
    lectura.to_excel(w, sheet_name='Archivos_leidos', index=False)
print('✔ Salidas en', C.CARPETA_SALIDA)

## 11. Interfaz web (objetivo 3)
Se escribe `app.py` y `LIMITACIONES.md`; luego se lanza Streamlit y se expone con un túnel. La pestaña **Licitaciones** consulta SECOP II desde la propia interfaz.

In [ ]:
%%writefile /content/bioprice_bi/app.py
"""Interfaz web (objetivo 3).  Ejecutar:  streamlit run app.py
En Colab:  !streamlit run app.py & npx localtunnel --port 8501   (o ejecútelo en su computador)"""
import json

import pandas as pd
import plotly.express as px
import streamlit as st

from bioprice import config as C
from bioprice import indicadores as I
from bioprice import ml, secop

st.set_page_config(page_title="BioPrice BI", layout="wide")
NOMBRE = {k: v["nombre"] for k, v in C.EQUIPOS.items()}


@st.cache_data(show_spinner="Cargando base analítica…")
def cargar():
    return pd.read_parquet(C.BASE_ANALITICA)


@st.cache_data(show_spinner="Consultando SECOP II…", ttl=3600)
def licitaciones(familias, limite):
    partes = []
    for ds in (C.SECOP_PROCESOS, C.SECOP_CONTRATOS):
        try:
            partes.append(secop.descargar(ds, familias=list(familias), limite=limite))
        except Exception as e:
            st.warning(f"No se pudo leer {ds}: {e}")
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()


try:
    base = cargar()
except FileNotFoundError:
    st.error(f"No existe {C.BASE_ANALITICA}. Ejecute primero `python run_pipeline.py`.")
    st.stop()

# ----------------------------------------------------------------- filtros
st.sidebar.title("Filtros")
fams = st.sidebar.multiselect("Familia de equipo", list(NOMBRE), default=list(NOMBRE), format_func=NOMBRE.get)
anios = sorted(base["ANIO"].dropna().astype(int).unique())
rango = st.sidebar.slider("Años", min(anios), max(anios), (min(anios), max(anios))) if len(anios) > 1 else (anios[0], anios[0])
paises = sorted(base["_pais"].dropna().unique())
sel_pais = st.sidebar.multiselect("País de origen", paises)
conf = st.sidebar.selectbox("Confianza mínima", ["Bajo", "Medio", "Alto"], index=0,
                            help="Alto = «Confirmado»: ficha técnica + INVIMA + peso compatible. Lo demás son candidatos.")
sin_anom = st.sidebar.checkbox("Excluir precios anómalos (ML)", value=True)
imp = st.sidebar.text_input("Importador contiene")
freq = st.sidebar.radio("Periodo", ["Y", "Q", "M"], format_func={"Y": "Año", "Q": "Trimestre", "M": "Mes"}.get, horizontal=True)

d = base[base["EQUIPO_CLASIFICADO"].isin(fams) & base["ANIO"].between(*rango)]
if sel_pais:
    d = d[d["_pais"].isin(sel_pais)]
if imp:
    d = d[d["_importador"].str.contains(imp.upper(), na=False)]
orden = {"Bajo": 0, "Medio": 1, "Alto": 2}
d = d[d["NIVEL_DE_CONFIANZA"].map(orden) >= orden[conf]]
if sin_anom:
    d = d[~d["ES_ANOMALIA_PRECIO"].fillna(False)]
d = d.assign(FAMILIA=d["EQUIPO_CLASIFICADO"].map(NOMBRE))

st.title("BioPrice BI · Precios de importación de equipo biomédico (Colombia)")
c1, c2, c3, c4 = st.columns(4)
c1.metric("Registros", f"{len(d):,}")
c2.metric("Valor FOB (USD)", f"{d['_fob'].sum():,.0f}")
c3.metric("Precio unitario mediano (USD)", f"{d['PRECIO_UNIT_USD'].median():,.0f}" if d["PRECIO_UNIT_USD"].notna().any() else "n/d")
c4.metric("Confirmados (Alto)", f"{int((d['ESTADO_IDENTIFICACION'] == 'Confirmado').sum()):,}")
if len(d) == 0:
    st.info("Los filtros no dejan registros.")
    st.stop()

t1, t2, t3, t4, t5, t6, t7 = st.tabs(["Precios", "Países", "Importadores", "Licitaciones", "Calidad y ML", "Datos", "Limitaciones"])

with t1:
    s = I.serie_precios(d, freq, por=("FAMILIA",))
    st.plotly_chart(px.line(s, x="PERIODO", y="mediana", color="FAMILIA", markers=True, hover_data=["n", "p25", "p75", "VAR_PCT_MEDIANA"],
                            title="Precio unitario mediano (USD)", log_y=st.checkbox("Escala logarítmica", True)), use_container_width=True)
    st.plotly_chart(px.bar(s, x="PERIODO", y="VAR_PCT_MEDIANA", color="FAMILIA", barmode="group", title="Variación % de la mediana vs periodo anterior"),
                    use_container_width=True)
    st.plotly_chart(px.box(d.dropna(subset=["PRECIO_UNIT_USD"]), x="FAMILIA", y="PRECIO_UNIT_USD", points="outliers", log_y=True,
                           title="Distribución del precio unitario"), use_container_width=True)
    st.caption("Periodos con menos de 5 registros (columna MUESTRA_PEQUENA) son poco fiables.")
    st.dataframe(s, use_container_width=True)

with t2:
    p = I.comparacion_pais(d.assign(EQUIPO_CLASIFICADO=d["FAMILIA"]))
    fam_p = st.selectbox("Familia", sorted(p["EQUIPO_CLASIFICADO"].unique()), key="fam_pais")
    pf = p[p["EQUIPO_CLASIFICADO"] == fam_p]
    st.plotly_chart(px.bar(pf, x="PAIS_ORIGEN", y="mediana", color="PRIMA_PCT_VS_FAMILIA", hover_data=["n", "CUOTA_UNIDADES_PCT"],
                           color_continuous_scale="RdBu_r", color_continuous_midpoint=0,
                           title="Precio mediano por país de origen (color = prima % vs mediana de la familia)"), use_container_width=True)
    st.dataframe(pf, use_container_width=True)

with t3:
    dd = d.assign(EQUIPO_CLASIFICADO=d["FAMILIA"])
    perfil = ml.segmentar_importadores(I.perfil_importadores(dd))
    fam_i = st.selectbox("Familia", sorted(perfil["EQUIPO_CLASIFICADO"].unique()), key="fam_imp")
    pi = perfil[perfil["EQUIPO_CLASIFICADO"] == fam_i].head(25)
    st.plotly_chart(px.bar(pi, x="valor_fob", y="IMPORTADOR", color="SEGMENTO", orientation="h", title="Principales importadores por valor FOB (USD)")
                    .update_yaxes(autorange="reversed"), use_container_width=True)
    st.plotly_chart(px.area(I.evolucion_cuotas(dd, fam_i), x="ANIO", y="CUOTA_PCT", color="NOMBRE", title="Evolución de la cuota de valor"),
                    use_container_width=True)
    conc = I.concentracion(dd)
    st.plotly_chart(px.line(conc[conc["EQUIPO_CLASIFICADO"] == fam_i], x="ANIO", y=["HHI", "TOP5_PCT"], markers=True,
                            title="Concentración del mercado (HHI y cuota del top-5)"), use_container_width=True)
    st.dataframe(pi, use_container_width=True)

with t4:
    st.write("Procesos y contratos de SECOP II relacionados con las familias. **El valor es del proceso/contrato, no precio unitario.**")
    fam_l = st.multiselect("Familias a consultar", list(NOMBRE), default=list(NOMBRE), format_func=NOMBRE.get, key="fam_lic")
    lim = st.number_input("Máx. registros por dataset", 100, 20000, 3000, step=500)
    if st.button("Consultar SECOP II") and fam_l:
        st.session_state["lic"] = licitaciones(tuple(fam_l), int(lim))
    lic = st.session_state.get("lic")
    if lic is not None and len(lic):
        lic = lic[lic["CATEGORIA_LICITACION"] == st.radio("Categoría", ["Equipo principal completo", "Accesorio", "Consumible", "Repuesto", "Indeterminado"],
                                                          horizontal=True)]
        st.plotly_chart(px.bar(lic.groupby(["ANIO", "FAMILIA"])[["VALOR_ESTIMADO", "VALOR_ADJUDICADO"]].sum().reset_index(), x="ANIO",
                               y="VALOR_ADJUDICADO", color="FAMILIA", title="Valor adjudicado por año (COP)"), use_container_width=True)
        st.dataframe(lic, use_container_width=True)

with t5:
    try:
        met = json.load(open(C.CARPETA_SALIDA / "metricas_ml.json", encoding="utf-8"))
        st.json(met)
    except FileNotFoundError:
        st.info("Sin métricas de ML.")
    st.plotly_chart(px.histogram(d, x="NIVEL_DE_CONFIANZA", color="FAMILIA", barmode="group", title="Nivel de confianza por familia"), use_container_width=True)
    an = base[base["ES_ANOMALIA_PRECIO"].fillna(False)]
    st.write(f"Registros marcados como anómalos de precio (todo el periodo): {len(an):,}")
    st.dataframe(an[["FAMILIA" if "FAMILIA" in an else "EQUIPO_CLASIFICADO", "_importador", "_pais", "PRECIO_UNIT_USD", "PRECIO_ESPERADO_USD",
                     "RAZON_PRECIO_ESPERADO", "METODO_ANOMALIA", "OBSERVACIONES"]].head(500), use_container_width=True)

with t6:
    cols = ["FECHA", "FAMILIA", "_importador", "_pais", "MODELO_CANDIDATO", "_cantidad", "_fob", "PRECIO_UNIT_USD", "PESO_UNITARIO_KG",
            "NIVEL_DE_CONFIANZA", "ESTADO_IDENTIFICACION", "VERIFICACION_INVIMA", "URL_FICHA", "OBSERVACIONES"]
    st.dataframe(d[[c for c in cols if c in d.columns]].head(5000), use_container_width=True)
    st.download_button("Descargar (CSV)", d.to_csv(index=False).encode("utf-8-sig"), "bioprice_filtrado.csv")

with t7:
    try:
        st.markdown(open("LIMITACIONES.md", encoding="utf-8").read())
    except FileNotFoundError:
        st.info("Falta LIMITACIONES.md")

%%writefile /content/bioprice_bi/LIMITACIONES.md
# Limitaciones conocidas

**Datos**
- Las subpartidas y los rangos de peso genéricos son criterios **preliminares**; no se han verificado contra el arancel oficial de la DIAN. Una subpartida puede agrupar equipos distintos (p. ej. 9019.20 incluye oxigenoterapia y CPAP; 9402.90 es mobiliario médico en general).
- El precio unitario es **FOB / cantidad** y solo se calcula si la unidad comercial es individual. Es valor de aduana en USD, no el precio que paga el hospital (no incluye margen del distribuidor, instalación ni servicio).
- Un mismo registro puede incluir varios componentes o accesorios; el peso neto y el FOB pueden corresponder a la línea completa, no solo al equipo.
- Las descripciones aduaneras son texto libre: marcas y modelos mal escritos o ausentes dejan el registro como *Candidato*.
- Si cambian los nombres de columnas de la fuente, hay que ajustar `ALIAS_COL`/`COL_MANUAL` en `config.py`.

**Verificación**
- Solo *Alto* = *Confirmado*: equipo completo + subpartida + (INVIMA o registro citado) + peso compatible con la ficha. Sin ficha técnica o sin INVIMA ningún registro llega a Confirmado.
- El archivo del INVIMA contiene solo registros vigentes hoy y no trae referencia/modelo: «no encontrado» es **no concluyente** y no penaliza.
- La tolerancia de ±25 % sobre el peso de ficha es una decisión editable (`TOLERANCIA_PESO_FICHA`), no una norma.
- La cobertura de fichas depende de las marcas/modelos cargados en el Excel; modelos sin ficha quedan sin peso verificable.

**Machine learning**
- Anomalías de precio: marcan valores atípicos **estadísticamente**, no errores ni fraude; revíselos antes de excluirlos.
- Precio esperado: se evalúa con validación cruzada; solo es útil si su error es menor que el de la línea base (se reporta en `metricas_ml.json`). Con pocos datos por familia se desactiva.
- Clasificador de texto: se entrena con etiquetas *débiles* (las propias reglas), por lo que su F1 mide acuerdo con las reglas, no exactitud real. Valídelo con una muestra revisada a mano.
- Segmentación de importadores (KMeans): agrupa por tamaño y recurrencia; el número de grupos es una elección del analista.

**Licitaciones (SECOP II)**
- El valor es del proceso o contrato, no precio unitario; puede incluir varias unidades, servicios o mantenimiento.
- Las columnas se descubren desde los metadatos del dataset; los nombres usados en `CANDIDATOS` deben revisarse la primera vez. La cobertura es la de SECOP II (no incluye todo el gasto público ni el privado).
- La relación con importaciones es indicativa: no existe un identificador común entre una licitación y una declaración de importación.

**Interfaz**
- Las gráficas se calculan sobre la base analítica del último `run_pipeline.py`; no se actualiza sola.
- Periodos o países con pocos registros se marcan (`MUESTRA_PEQUENA`) y no deben interpretarse como tendencias.

In [ ]:
# Ejecute esta celda y abra la URL que imprime (la contraseña del túnel es la IP pública que se muestra)
import subprocess, urllib.request, time
os.environ['BIOPRICE_BASE'] = str(C.BASE)
print('IP del túnel (contraseña):', urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip())
subprocess.Popen(['streamlit', 'run', '/content/bioprice_bi/app.py', '--server.port', '8501', '--server.headless', 'true'], cwd='/content/bioprice_bi')
time.sleep(5)
!npx -y localtunnel --port 8501